# Can bank reports help us decide which banks to examine first?

**Why examine a bank’s deposits?**

Customers expect their bank to complete payments and withdrawals when needed. A bank therefore needs cash and other reliable ways to obtain cash.

Some assets are loans that customers will repay over years. Payments and withdrawals can arrive much sooner. When deposits leave, the bank may need to use available cash, sell assets, or obtain replacement funding. Those choices can carry costs.

**That makes deposit changes worth understanding.** A person examining the bank would also consider its available funding, assets, customers, and institutional history.

**So here is the question. Can bank reports help that person decide where to look first?**

### A made-up USD 100 bank

For one teaching example, give the bank **USD 100 in assets**. It has **USD 15 in cash**. Borrowers owe it **USD 85 in loans**, which they will repay over time.

Now, in this invented example, a customer asks to withdraw **USD 20**.

**How much extra cash does this example need?** **USD 5.** The first visual separates money available today from loans repaid later. These numbers are not a real bank, a recorded withdrawal, or a forecast.

Keep the same made-up bank for one more step. It owes customers USD 90 in deposits, so USD 10 is equity. That explains the balance-sheet difference. It does not add USD 10 to the cash pile.

The example isolates cash availability. Lending can itself create deposits; the [Bank of England explains that mechanism](https://www.bankofengland.co.uk/quarterly-bulletin/2014/q1/money-creation-in-the-modern-economy). The [Federal Reserve’s funding study](https://www.federalreserve.gov/econres/notes/feds-notes/assessing-bank-resilience-to-a-funding-shock-20260217.html) explains how replacement funding can raise costs in a modeled funding shock. Neither source evaluates this project's forecasts.

**The answer from the original experiment:** assume time to examine **10 of every 100 banks**. Random selection would find about **one** bank whose next-quarter deposit growth finishes in the lowest 10%. Ridge and the neural network found about **two**, averaged across three historical quarters of 2024. A human would investigate the movement and decide whether it needs attention. The 10% examination capacity is hypothetical; no operational savings were measured.

**Project in one minute.** The code learns from 214,425 earlier bank-quarter examples. Predicting zero growth has the lowest average absolute error among the original four models; the network has the lowest RMSE, which gives larger mistakes more weight. The review-list comparison asks a separate question: how many low-growth outcomes enter a fixed-size list? The expanded follow-up comparison gives the seasonal-median rule the lowest MAE: **3.500 percentage points**, versus **3.601** for zero growth. This rule was added after 2024 had been examined. Ridge’s review precision exceeds size-only Ridge in the conditional bank-resampling interval; the MLP–Ridge interval includes either ordering. Detailed checks follow the original conclusion.

### Twelve years of reports have three different jobs

| Job | Predictor reports | Bank-quarter examples |
|---|---|---:|
| Learn patterns | Q2 2013–Q3 2022: **38 quarters** | **214,425** |
| Choose settings | Q1–Q3 2023: **3 quarters** | **13,834** |
| Measure historical performance | Q1–Q3 2024: **3 quarters** | **13,532** |

The source contains **48 quarters of reports, 2013–2024**. Most earlier reports have a next-quarter answer, subject to bank-specific gaps. We assign earlier periods to learning and model selection. The previous-quarter input starts the usable training examples in June 2013.

**Why three evaluation quarters?** March 2024 predicts June; June predicts September; September predicts December. A December 2024 forecast would need March 2025, outside this dataset. December 2022 and December 2023 predictor rows are omitted so their outcomes do not cross into the following stage.

**How fresh is the evaluation?** The core models fit earlier data and use 2023 to choose settings. The project has examined 2024 repeatedly. Later comparisons were developed with those results known, so we call this a **reused historical evaluation period**. Thousands of bank examples share three evaluation dates. A later untouched period would test performance under new conditions.

**Tools:** Python, pandas, scikit-learn, TensorFlow, Plotly, Jupyter, and uv. The foundation-model extensions use PyTorch and MLX.




In [ ]:
NOTEBOOK_EDITION = "submission"
READING_GUIDES = {}

In [ ]:
# Keep imports, frozen settings, and rendering tools together.
import os, sys, json, hashlib, platform, time, html, textwrap, io
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("MPLCONFIGDIR", str(Path.cwd() / ".mpl-cache"))
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import get_plotlyjs
from IPython.display import display, HTML
import tensorflow as tf
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from wm_notecards import WMTheme, init_notebook
from wm_notecards.cards import (
    preview_card,
    big_number_card,
    question_card,
    takeaway_card,
    wm_counterintuitive_card,
    wm_formula_card,
)
from wm_notecards.charts import (
    plot_shell_html,
    style_fig_wm,
    wm_render_figure_card,
)
from wm_notecards.pictogram import pictogram_card
from wm_notecards.eda import display_data_chips, wm_compare_fields
from wm_notecards.tables import (
    display_cols_by_dtype,
    style_describe_wm,
    wm_render_micro_profile_cards,
    wm_render_styler,
)

SEED = 42
FEATURES = [
    "log_deposits",
    "cash_ratio",
    "loan_ratio",
    "equity_ratio",
    "prior_growth",
]
MODEL_COLORS = {
    "Zero growth": "#627381",
    "Persistence": "#A86223",
    "Ridge": "#3F6294",
    "MLP": "#0B6F75",
}
MODEL_ORDER = ["Zero growth", "Persistence", "Ridge", "MLP"]
SIZE_ORDER = ["Smallest", "Lower middle", "Upper middle", "Largest"]
SLICE_ORDER = ["All", "Realized bottom 25%", "Realized bottom 10%"]

ROOT = Path.cwd()
assert (ROOT / "data/fdic_financials_2020_2024.csv").is_file(), "Run from the project folder."

OUT = ROOT / "growth_outputs" / NOTEBOOK_EDITION
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "charts").mkdir(exist_ok=True)

In [ ]:
tf.config.set_visible_devices([], "GPU")
tf.config.threading.set_inter_op_parallelism_threads(2)
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.experimental.enable_op_determinism()
tf.keras.utils.set_random_seed(SEED)

# Use the repository's standard card, chip, typography, and accent colors.
# White plotting surfaces match the user's preferred white cards.
theme = WMTheme(width=860, height=480, card_bg="#FFFFFF", plot_bg="#FFFFFF")
init_notebook(expand_colab_outputs=True)

# VS Code places native Plotly outputs against the left edge of the output
# region. Center every renderer output once here so individual charts cannot
# drift away from the card column.
display(
    HTML(
        """
        <style>
        body { background: #EFF1F6; color: #172F3E; }
        .jp-RenderedMarkdown { color: #172F3E; }
        .jp-RenderedMarkdown h1, .jp-RenderedMarkdown h2, .jp-RenderedMarkdown h3 { color: #172F3E; }
        .jp-RenderedMarkdown a { color: #3F6294; }
        .jp-RenderedMarkdown { max-width: 940px; margin: auto; line-height: 1.65; }

        .output_container .output {
            display: flex !important;
            justify-content: center !important;
        }
        .output_container .output > div {
            margin-left: auto !important;
            margin-right: auto !important;
        }
        .wm-micro-rail {
            max-width: 860px !important;
            grid-auto-flow: row !important;
            grid-template-columns: repeat(2, minmax(0, 1fr)) !important;
            grid-auto-columns: auto !important;
            overflow-x: visible !important;
        }
        .wm-micro-rail .wm-micro-card:last-child {
            grid-column: 1 / -1;
        }
        @media (max-width: 720px) {
            .wm-micro-rail {
                grid-template-columns: minmax(0, 1fr) !important;
            }
            .wm-micro-rail .wm-micro-card:last-child {
                grid-column: auto;
            }
            .wm-table-card:has(td.col4) {
                overflow-x: auto !important;
            }
            .wm-table-card:has(td.col4) table {
                min-width: 700px !important;
            }
            .wm-table-card:has(td.col6) table {
                min-width: 780px !important;
            }
        }
        </style>
        """
    )
)

In [ ]:
# One inline library makes saved chart outputs independent of a CDN.
display(HTML("<script>" + get_plotlyjs() + "</script>"))

In [ ]:
# Shared display helpers keep all plots centered and all exact receipts wrapped.
def table(frame, title, formats=None, wrap_columns=None):
    """Render a dataframe with the same centered teaching-card treatment."""
    default_formats = {
        column: (lambda value: value.strftime("%Y-%m-%d"))
        for column in frame.select_dtypes(include=["datetime"])
    }
    default_formats.update(
        {column: "{:,.3f}" for column in frame.select_dtypes(include=["floating"])}
    )
    default_formats.update(formats or {})

    styled = frame.style.hide(axis="index").format(
        default_formats,
        na_rep="Missing",
    )
    default_wrap = {column: 290 for column in frame if pd.api.types.is_string_dtype(frame[column])}
    default_wrap.update(wrap_columns or {})
    styled = styled.set_table_styles(
        [
            {
                "selector": f"td.col{frame.columns.get_loc(column)}",
                "props": [
                    ("white-space", "normal !important"),
                    ("overflow-wrap", "anywhere !important"),
                    ("word-break", "normal !important"),
                    ("text-align", "left"),
                ],
            }
            for column in default_wrap
        ],
        overwrite=False,
    )
    wm_render_styler(
        styled,
        theme=theme,
        title=title,
        wrap_columns=default_wrap,
    )


def ordered_rows(frame, columns, category_orders=None, ascending=True):
    """Return rows in the order a reader expects to scan them."""
    result = frame.copy()

    for column, values in (category_orders or {}).items():
        result[column] = pd.Categorical(
            result[column],
            categories=values,
            ordered=True,
        )

    return result.sort_values(
        columns,
        ascending=ascending,
        kind="stable",
    ).reset_index(drop=True)

In [ ]:
def chart(fig, name, title, subtitle="", height=540, legend_y=-0.24):
    """Apply one visual system and one centered renderer to every chart."""
    default_palette = {
        "#636efa": "#3F6294",
        "#EF553B": "#A86223",
        "#00cc96": "#0B6F75",
        "#ab63fa": "#76528B",
        "#FFA15A": "#A86223",
        "#19d3f3": "#0B6F75",
    }
    for trace in fig.data:
        for component in ["marker", "line"]:
            obj = getattr(trace, component, None)
            if obj is not None and isinstance(getattr(obj, "color", None), str):
                obj.color = default_palette.get(obj.color, obj.color)
    title = title.replace(chr(36), "USD ")
    subtitle = subtitle.replace(chr(36), "USD ")
    style_fig_wm(
        fig,
        title=title,
        subtitle=subtitle,
        theme=theme,
        normalize_legacy_colors=False,
        category_policy="preserve",
        allow_dense_categories=True,
    )

    title_html = (
        "<b>" + "<br>".join(html.escape(line) for line in textwrap.wrap(title, 52)) + "</b>"
    )
    if subtitle:
        wrapped_subtitle = "<br>".join(html.escape(line) for line in textwrap.wrap(subtitle, 88))
        title_html += '<br><span style="font-size:14px">' + wrapped_subtitle + "</span>"

    fig.update_layout(
        width=860,
        height=height,
        font=dict(size=14, family="Inter, Arial, sans-serif", color=theme.text_main),
        title=dict(
            text=title_html,
            font=dict(size=24),
            x=0.035,
            y=0.98,
            xanchor="left",
            yanchor="top",
        ),
        hovermode="closest",
        hoverlabel=dict(
            bgcolor="white",
            bordercolor="#B8C2CC",
            font=dict(
                size=14,
                color="#172F3E",
                family="Inter, Arial, sans-serif",
            ),
        ),
        legend=dict(
            y=legend_y,
            yanchor="top",
            x=0.5,
            xanchor="center",
            orientation="h",
            font=dict(size=14, family="Inter, Arial, sans-serif"),
            title_text="",
        ),
        margin=dict(l=85, r=45, t=155, b=155 if legend_y < -0.24 else 120),
        paper_bgcolor=theme.card_bg,
        plot_bgcolor=theme.plot_bg,
    )
    fig.update_xaxes(
        automargin=True,
        showline=True,
        linecolor="#ADB5BD",
        gridcolor="#ECEFF1",
        tickfont=dict(size=14, family="Inter, Arial, sans-serif"),
        title_font=dict(size=15, family="Inter, Arial, sans-serif"),
    )
    fig.update_yaxes(
        automargin=True,
        showline=True,
        linecolor="#ADB5BD",
        gridcolor="#ECEFF1",
        tickfont=dict(size=14, family="Inter, Arial, sans-serif"),
        title_font=dict(size=15, family="Inter, Arial, sans-serif"),
    )
    fig.for_each_yaxis(lambda axis: axis.update(dtick=1) if axis.type == "log" else None)
    fig.write_json(OUT / "charts" / f"{name}.json")

    if "google.colab" in sys.modules:
        wm_render_figure_card(fig, theme=theme, file_stub=name)
        return

    # VS Code chooses Plotly's native MIME renderer before the centered WM
    # shell. Render one self-contained HTML output locally so the shell owns
    # alignment. Plotly.js was loaded once in the setup cell above.
    figure_html = fig.to_html(
        full_html=False,
        include_plotlyjs=False,
        config={
            "displaylogo": False,
            "responsive": False,
            "displayModeBar": False,
        },
        default_width="860px",
        default_height=f"{height}px",
    )
    shell = plot_shell_html(
        figure_html,
        theme,
        figure_width=860,
    )
    display(HTML(shell))
    guide = READING_GUIDES.get(name)
    if guide:
        display(
            HTML(
                '<p style="max-width:860px;margin:12px auto 28px;line-height:1.6;text-align:left"><strong>Read this chart:</strong> '
                + html.escape(guide)
                + "</p>"
            )
        )

In [ ]:
def takeaway(title, body, metric=None):
    """Show the answer immediately after its evidence."""
    takeaway_card(title=title, body=body, metric=metric, theme=theme)


def scores(actual, predicted):
    """Report ordinary-growth errors in percentage points."""
    return {
        "MAE (pp)": 100 * mean_absolute_error(actual, predicted),
        "RMSE (pp)": 100 * np.sqrt(mean_squared_error(actual, predicted)),
    }


def log_scores(y_log, pred_log):
    """Score log-growth forecasts in ordinary percentage-point units."""
    return scores(np.expm1(y_log), np.expm1(pred_log))

In [ ]:
READING_GUIDES['cash_balance_sheet'] = 'This is an invented USD 100 example. The blue USD 15 is cash available now; the other USD 85 represents loans that borrowers will repay over time.'
# EXEMPLAR: formula-card
# Imagine 100 dollars of assets. Separate cash today from repayment later.
illustration = pd.DataFrame(
    {
        "Item": ["Cash", "Loans", "Customer deposits", "Equity"],
        "Dollars": [15, 85, 90, 10],
        "Meaning": [
            "Available for payment",
            "Repaid over time",
            "Money owed to customers",
            "Assets minus liabilities",
        ],
    }
)
fig = go.Figure()
for amount, left, label, color in [
    (15, 0, "Cash now: USD 15", "#3F6294"),
    (85, 15, "Loans repaid over time: USD 85", "#C5CDD6"),
]:
    fig.add_trace(
        go.Bar(
            x=[amount],
            base=[left],
            y=["Assets"],
            orientation="h",
            text=[label],
            textposition="inside",
            marker_color=color,
            showlegend=False,
        )
    )
fig.update_layout(barmode="overlay")
fig.update_xaxes(range=[0, 100], title="Dollars of assets")
chart(
    fig,
    "cash_balance_sheet",
    "Example: a bank with USD 100 worth of assets",
    "Invented numbers for one balance-sheet example. USD 15 is cash today; USD 85 is loans repaid later.",
    height=410,
)

In [ ]:
READING_GUIDES['withdrawal_shortfall'] = 'This is an invented request, not a real withdrawal. Blue supplies USD 15. Amber marks the USD 5 gap in this example.'
# The request arrives before the loans have been repaid.
cash_available, withdrawal_request = 15, 20
cash_shortfall = withdrawal_request - cash_available
assert cash_shortfall == 5
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=[cash_available],
        y=["How to pay"],
        orientation="h",
        text=["15 available"],
        textposition="inside",
        marker_color="#3F6294",
        name="Cash available",
    )
)
fig.add_trace(
    go.Bar(
        x=[cash_shortfall],
        y=["How to pay"],
        orientation="h",
        text=["5 needed"],
        textposition="inside",
        marker_color="#A86223",
        name="Additional cash",
    )
)
fig.update_layout(barmode="stack", showlegend=False)
fig.update_xaxes(range=[0, 20], title="Invented customer request: USD 20")
chart(
    fig,
    "withdrawal_shortfall",
    "Example: USD 20 requested, USD 15 available, USD 5 to obtain",
    "Invented numbers. Blue is cash already available. Amber is the extra cash this example needs.",
    height=410,
)
takeaway(
    "In this example, the bank still needs USD 5 in cash",
    "This is a teaching example, not a real withdrawal or a claim about a real bank. "
    "The USD 85 in loans is valuable, but it may be repaid years from now. "
    "For this example, the remaining USD 5 could come from borrowing or selling an asset. The cost and availability would need evidence.",
)

In [ ]:
# Now introduce how the assets are financed, after the cash problem is clear.
question_card(
    title="Okay, so where is equity in our made-up bank?",
    theme=theme,
    body="We are still using the same invented USD 100 example. The bank owes customers USD 90 in deposits. "
    "That leaves USD 10 of equity. The cash asset is still USD 15. Equity explains the difference. "
    "It does not create another USD 10 of cash.",
)
wm_formula_card(
    title="So what fills the USD 5 gap in this example?",
    subtitle="Still invented numbers. Equity explains the balance-sheet difference. It is not another pile of cash.",
    theme=theme,
    items=[
        {
            "label": "Cash gap",
            "latex": r"\$20 - \$15 = \$5",
            "fallback": "USD 20 requested minus USD 15 cash available equals USD 5 to obtain.",
        },
        {
            "label": "Equity",
            "latex": r"\$100 - \$90 = \$10",
            "fallback": "USD 100 in assets minus USD 90 in deposits equals USD 10 in equity.",
        },
        {
            "label": "After borrowing and paying",
            "latex": r"\$85 = \$70 + \$5 + \$10",
            "fallback": "USD 85 in assets equals USD 70 in deposits, USD 5 in borrowing, and USD 10 in equity.",
        },
    ],
)
wm_counterintuitive_card(
    title="Can the USD 10 equity balance pay the withdrawal?",
    theme=theme,
    why_misread="Equity sounds like a separate pile of spare cash.",
    ordinary_process="Equity is the difference between assets and liabilities. In this example the cash asset is USD 15; the loans total USD 85.",
    conclusion_boundary="The bank needs another USD 5 of cash, for example from borrowing or selling an asset. Equity absorbs losses if assets lose value. Funding cost and availability require their own investigation.",
    kicker="Cash and loss absorption",
    chip_text="LOOK TWICE",
)

In [ ]:
# EXEMPLAR: purposeful-source-preview
# Choose a reproducible illustration near the median asset size, without looking at future growth.
bridge_source = pd.read_csv(ROOT / "data/fdic_financials_2013_2024.csv")
bridge_saved = pd.read_csv(ROOT / "growth_outputs/masterclass/predictions.csv")
bridge_certificates = bridge_saved.loc[bridge_saved.date.eq("2024-03-31"), "CERT"]
bridge_march = bridge_source.loc[
    bridge_source.REPDTE.eq(20240331) & bridge_source.CERT.isin(bridge_certificates)
].copy()
bridge_march["distance_from_median"] = (bridge_march.ASSET - bridge_march.ASSET.median()).abs()
bridge_bank = bridge_march.sort_values(["distance_from_median", "CERT"]).iloc[0]
bridge_prior = bridge_source.loc[
    bridge_source.CERT.eq(bridge_bank.CERT) & bridge_source.REPDTE.eq(20231231)
].iloc[0]
bridge_other_assets = bridge_bank.ASSET - bridge_bank.CHBAL - bridge_bank.LNLSNET
assert bridge_other_assets >= 0
question_card(
    title="How does a real bank report become five model inputs?",
    theme=theme,
    body=f"Meet {bridge_bank.NAME}, CERT {int(bridge_bank.CERT)}, in March 2024. "
    "We chose the eligible report closest to the median asset size, breaking ties by CERT. "
    "That rule uses current assets, without choosing an unusually good or bad future outcome. "
    "The illustration comes from the historical evaluation population; a prospective trial needs its own scoring-time eligibility rules.",
)

In [ ]:
READING_GUIDES['real_bank_assets'] = 'The entire bar is the bank’s reported assets. Blue marks cash and balances due; the other segments show net loans and the remaining assets. The exact receipt below supplies values for the small cash segment.'
# Source balances are in thousands of US dollars. Divide by 1,000 to show millions.
bridge_assets = pd.DataFrame(
    {
        "Asset": ["Cash and balances due", "Net loans and leases", "Other assets (remainder)"],
        "USD million": [
            bridge_bank.CHBAL / 1000,
            bridge_bank.LNLSNET / 1000,
            bridge_other_assets / 1000,
        ],
    }
)
fig = go.Figure()
for (_, item), color in zip(bridge_assets.iterrows(), ["#3F6294", "#A8BBC8", "#D9DFE3"]):
    fig.add_trace(
        go.Bar(
            x=[item["USD million"]],
            y=["Reported assets"],
            orientation="h",
            name=item.Asset,
            marker_color=color,
            hovertemplate=item.Asset + ": USD %{x:.3f}M<extra></extra>",
        )
    )
fig.update_layout(barmode="stack")
fig.update_xaxes(title="USD millions", range=[0, bridge_bank.ASSET / 1000])
chart(
    fig,
    "real_bank_assets",
    f"{bridge_bank.NAME}: USD {bridge_bank.ASSET / 1000:.2f}M in assets",
    "March 31, 2024; cash, net loans, and the remaining reported assets",
    height=440,
)
display(bridge_assets.round(3))
takeaway(
    "The same division turns different-sized banks into comparable ratios",
    f"This report lists USD {bridge_bank.CHBAL / 1000:.3f}M in cash and balances due and USD {bridge_bank.ASSET / 1000:.3f}M in assets. "
    f"Divide the first by the second: cash/assets is {bridge_bank.CHBAL / bridge_bank.ASSET:.2%}. "
    "Other assets matter: real balance sheets contain more than cash and loans. "
    "CHBAL is a reported accounting category, not a complete assessment of immediately usable liquidity.",
)

In [ ]:
# Map the actual report to exactly the five original model features.
bridge_inputs = pd.DataFrame(
    [
        ["Deposit size", "log(DEPDOM in USD thousands)", np.log(bridge_bank.DEPDOM)],
        [
            "Cash / assets",
            f"{bridge_bank.CHBAL:,} / {bridge_bank.ASSET:,}",
            bridge_bank.CHBAL / bridge_bank.ASSET,
        ],
        [
            "Loans / assets",
            f"{bridge_bank.LNLSNET:,} / {bridge_bank.ASSET:,}",
            bridge_bank.LNLSNET / bridge_bank.ASSET,
        ],
        [
            "Equity / assets",
            f"{bridge_bank.EQ:,.0f} / {bridge_bank.ASSET:,}",
            bridge_bank.EQ / bridge_bank.ASSET,
        ],
        [
            "Previous deposit growth",
            f"log({bridge_bank.DEPDOM:,} / {bridge_prior.DEPDOM:,})",
            np.log(bridge_bank.DEPDOM / bridge_prior.DEPDOM),
        ],
    ],
    columns=["Input", "Calculation from report", "Value before scaling"],
)
table(
    bridge_inputs,
    "Five inputs from this report and its previous quarter",
    formats={"Value before scaling": "{:.5f}"},
)
bridge_inputs.to_csv(OUT / "real_bank_inputs.csv", index=False)
bridge_record = bridge_bank.drop(labels=["distance_from_median"]).to_frame().T
bridge_record["Previous deposits"] = bridge_prior.DEPDOM
bridge_record.to_csv(OUT / "real_bank_receipt.csv", index=False)
wm_formula_card(
    title="What would a hypothetical 10% deposit decline amount to?",
    theme=theme,
    items=[
        {
            "label": "Scenario, not an observed withdrawal",
            "fallback": f"10% × USD {bridge_bank.DEPDOM / 1000:.3f}M deposits = USD {bridge_bank.DEPDOM * 0.1 / 1000:.3f}M",
        },
        {
            "label": "Compare with the reported cash category",
            "fallback": f"USD {bridge_bank.DEPDOM * 0.1 / 1000:.3f}M / USD {bridge_bank.CHBAL / 1000:.3f}M = {bridge_bank.DEPDOM * 0.1 / bridge_bank.CHBAL:.2f} times",
        },
    ],
)
takeaway(
    "That comparison raises a funding question",
    "The hypothetical decline is larger than this report's cash category. Whether the bank could meet payments would depend on timing, asset sales, borrowing capacity, other funding, and the reason deposits changed. "
    "A quarterly deposit change is not a record of cash withdrawals. The model forecasts deposit growth; it does not run this hypothetical cash test or diagnose the bank.",
)

In [ ]:
READING_GUIDES['opening_review'] = 'Each bar counts selected banks that later fell in the lowest-growth group. All methods have 454 March selections. The random bar is an expectation, so its count can be fractional.'
# Read the saved original run for an early preview; fresh execution verifies it later.
opening_ranking = pd.read_csv(ROOT / "growth_outputs/masterclass/ranking.csv")
opening_march = opening_ranking.loc[opening_ranking["Quarter"].eq("2024-03-31")].set_index("Model")
opening_banks = int(opening_march.loc["Ridge", "Banks"])
opening_slots = int(opening_march.loc["Ridge", "Selected"])
opening_counts = pd.DataFrame(
    {
        "Selection": ["Random expectation", "Ridge", "Neural network"],
        "Hits": [
            opening_slots**2 / opening_banks,
            opening_march.loc["Ridge", "Hits"],
            opening_march.loc["MLP", "Hits"],
        ],
    }
)
fig = go.Figure(
    go.Bar(
        x=opening_counts["Hits"],
        y=opening_counts["Selection"],
        orientation="h",
        text=[
            f"{v:.2f} expected" if i == 0 else f"{int(v)} found"
            for i, v in enumerate(opening_counts["Hits"])
        ],
        textposition="outside",
        marker_color=["#C5CDD6", "#3F6294", "#0B6F75"],
    )
)
fig.update_xaxes(title="Selected banks later in the lowest-growth 10%", range=[0, 140])
fig.update_yaxes(autorange="reversed", title="")
chart(
    fig,
    "opening_review",
    f"{opening_banks:,} banks. {opening_slots:,} places to look.",
    "March 2024 reports → June outcomes; fixed 10% capacity is an experiment assumption",
    height=480,
)
takeaway(
    "Imagine you can examine only 10 of every 100 banks",
    "After the next quarter, identify the 10 banks with the lowest deposit growth. "
    "Selecting 10 banks randomly would include about one of those banks on average. "
    "The Ridge and neural-network lists included about two, averaged across our three historical quarters. "
    "This is a simplified explanation of measured rates. The chart gives the exact March counts. "
    "The same amount of review time brought more intended cases into the list. "
    "A person would investigate why their deposits changed; the lowest-growth group can even include positive growth.",
)

In [ ]:
# Repeat the decisive counts in cards, beside the full chart above.
question_card(
    title="Which 454 banks would you examine first?",
    theme=theme,
    body="The experiment sorts banks from lowest to highest predicted deposit growth. After June’s reports arrive, we count how many selections actually belong to that quarter’s lowest-growth tenth.",
    kicker="March 2024 example",
    chip_text="THE DECISION",
)
big_number_card(
    title="Ridge found 112 of the intended cases",
    theme=theme,
    value=f"{int(opening_march.loc['Ridge', 'Hits']):,}",
    value_label=f"Among {opening_slots:,} selected banks",
    body=f"Random selection: {opening_slots**2 / opening_banks:.2f} expected. Neural network: {int(opening_march.loc['MLP', 'Hits'])} found. March-to-June 2024; hypothetical 10% review capacity.",
)
pictogram_card(
    percent=float(opening_march.loc["Ridge", "Hits"]) / opening_slots,
    headline="About one in four Ridge selections reached the lowest-growth group",
    big_text=f"{int(opening_march.loc['Ridge', 'Hits'])} / {opening_slots}",
    icon="dot",
    theme=theme,
    subtitle="March-to-June 2024: 24.7% of selected banks. Filled dots show an approximate share; each dot is a unit of the grid, rather than an individual bank. A person still investigates the cause.",
    kicker="Historical selection precision",
)

In [ ]:
READING_GUIDES['history_roles'] = 'Read left to right through predictor dates. The long earlier interval is training. The two short later intervals choose settings and measure reused historical performance; their labels count bank-quarter examples.'
# Keep the source history and the model's three jobs visible at the beginning.
opening_periods = pd.DataFrame(
    {
        "Job": ["Learn patterns", "Choose settings", "Measure historical performance"],
        "First predictor": pd.to_datetime(["2013-06-30", "2023-03-31", "2024-03-31"]),
        "Last predictor": pd.to_datetime(["2022-09-30", "2023-09-30", "2024-09-30"]),
        "Predictor quarters": [38, 3, 3],
        "Examples": [214425, 13834, 13532],
    }
)
fig = go.Figure()
for position, row in opening_periods.iterrows():
    fig.add_trace(
        go.Scatter(
            x=[row["First predictor"], row["Last predictor"]],
            y=[position, position],
            mode="lines+markers",
            showlegend=False,
            line=dict(width=18, color=["#0B6F75", "#A86223", "#3F6294"][position]),
            marker=dict(size=10),
            hovertemplate=f"{row['Job']}: {row['Predictor quarters']} quarters; {row['Examples']:,} examples<extra></extra>",
        )
    )
fig.update_yaxes(
    tickvals=[0, 1, 2],
    ticktext=["38 quarters · learn", "3 quarters · choose", "3 quarters · evaluate"],
    autorange="reversed",
)
fig.update_xaxes(title="Predictor report date", range=["2013-01-01", "2025-01-01"])
chart(
    fig,
    "history_roles",
    "Twelve years of reports have three different jobs",
    "48 source quarters; 38 training predictor quarters; three validation and three reused evaluation quarters",
    height=500,
)
display(opening_periods)
(OUT / "opening_periods.csv").write_text(opening_periods.to_csv(index=False))

## 1 · What is in the 2013–2024 data?

**In plain terms:** **One row is one bank's report for one quarter.** Imagine finding the same bank in March and June. Those are two examples from one institution. CERT identifies the bank; REPDTE identifies the report date. Deposits, assets, loans, cash, and equity describe that report.

**Would a blank equity entry mean the bank has zero equity?** A blank means this extract gives us no number. A recorded zero supplies a number whose meaning still needs checking against the reporting definition. The 815 EQ blanks all fall outside our chosen domestic-bank/reporting-form population. We preserve the source blanks and show that exclusion beside the pandas output.

**Why begin in 2013?** We chose 2013–2024 because it follows the 2012 reporting conversion. Matching column names across older forms do not establish matching definitions. The older reports remain in the detailed audit. Read the column dictionary, a few actual rows, types, missing counts, and summaries before using a ratio.



In [ ]:
# EXEMPLAR: analytical-question
# Give the decision before the audit that supports it.
question_card(
    title="Can reports from 2013 and 2024 live in one experiment?",
    theme=theme,
    body=(
        "We use 2013–2024. Earlier reports cross a 2012 reporting-form change; "
        "their five-field historical mapping remains unverified. The newer "
        "48 quarters match the long file exactly."
    ),
    kicker="Data history",
    chip_text="QUESTION",
)


def ordered(frame):
    """Return bank-quarter rows in a stable order for exact comparisons."""
    return frame.sort_values(["CERT", "REPDTE"]).reset_index(drop=True)


def schema_table(frame):
    """Summarize the basic structure a reader checks at the start of EDA."""
    return pd.DataFrame(
        {
            "Column": frame.columns,
            "Data type": frame.dtypes.astype(str).to_numpy(),
            "Missing rows": frame.isna().sum().to_numpy(),
            "Distinct values": frame.nunique(dropna=False).to_numpy(),
        }
    )


# Keep the audit history separate from the post-conversion modeling history.
long_history = pd.read_csv(ROOT / "data/fdic_financials_2010_2024.csv")
post_conversion = pd.read_csv(ROOT / "data/fdic_financials_2013_2024.csv")
recent_extract = pd.read_csv(ROOT / "data/fdic_financials_2020_2024.csv")
reporting = pd.read_csv(ROOT / "data/fdic_reporting_2010_2024.csv")
keys = ["CERT", "REPDTE"]

# Teach the source vocabulary before asking the reader to inspect numbers.
source_dictionary = pd.DataFrame(
    [
        ("REPDTE", "Report date", "The quarter-end date for this report"),
        ("CERT", "Institution ID", "FDIC certificate; links reports for one institution"),
        ("NAME", "Institution name", "Reported name; can change over time"),
        ("STALP", "State abbreviation", "Location label; not a model input"),
        ("DEPDOM", "Domestic deposits", "Deposits in domestic offices; USD thousands"),
        ("ASSET", "Total assets", "Total reported assets; USD thousands"),
        ("LNLSNET", "Net loans and leases", "Reported net loan and lease balance; USD thousands"),
        (
            "CHBAL",
            "Cash and balances due",
            "Cash and balances due from depository institutions; USD thousands",
        ),
        ("EQ", "Equity capital", "Reported equity capital; USD thousands"),
    ],
    columns=["Column", "Plain name", "Meaning and units"],
)
table(source_dictionary, "Read the column names before reading the numbers")

In [ ]:
# First contact with the data: size, sample rows, types, missingness, and keys.
shape_receipt = pd.DataFrame(
    {
        "Rows": [len(post_conversion)],
        "Columns": [post_conversion.shape[1]],
        "Banks": [post_conversion["CERT"].nunique()],
        "Quarters": [post_conversion["REPDTE"].nunique()],
        "First quarter": [post_conversion["REPDTE"].min()],
        "Last quarter": [post_conversion["REPDTE"].max()],
        "Duplicate bank-quarters": [int(post_conversion.duplicated(keys).sum())],
    }
)
table(shape_receipt, "What is in the post-conversion extract?")
table(
    post_conversion[["REPDTE", "CERT", "NAME", "STALP"]].head(5),
    "Five source rows: identity and reporting date",
)
table(
    post_conversion[["CERT", "DEPDOM", "ASSET", "LNLSNET", "CHBAL", "EQ"]].head(5),
    "The same five rows: financial balances in USD thousands",
)
table(schema_table(post_conversion), "Column types, missing values, and cardinality")

# The familiar pandas checks come first. The cards below explain their results.
post_conversion.head()

In [ ]:
info_buffer = io.StringIO()
post_conversion.info(buf=info_buffer, memory_usage="deep")
print(info_buffer.getvalue())

In [ ]:
post_conversion.isna().sum().sort_values(ascending=False)

In [ ]:
# EXEMPLAR: missingness-evidence
# These two illustrative entries show why a missing value cannot be read as zero.
display(
    pd.DataFrame(
        {
            "Illustrative EQ entry": [np.nan, 0.0],
            "Reading": [
                "No value supplied in this extract",
                "A recorded numeric zero; verify its reporting meaning",
            ],
        }
    )
)
# Explain the missingness result while it is still on screen.
source_audit = post_conversion.merge(reporting, on=keys, validate="one_to_one")
missing_eq = source_audit["EQ"].isna()
inside_population = source_audit["BKCLASS"].isin(
    ["N", "NM", "SM", "SB", "SI", "SL"]
) & source_audit["CALLFORM"].isin([31, 41, 51])
assert not (missing_eq & inside_population).any()
assert source_audit.loc[missing_eq, "CALLFORM"].eq(2).all()
preview_card(
    title="Missing equity stays unknown; these reports are outside our population",
    theme=theme,
    body=f"The 2013–2024 source has {missing_eq.sum():,} missing EQ values. All occur on form 2; none belong to the domestic-bank/report-form group used for modeling.",
    bullets=[
        "1. A blank means unknown. We do not replace source equity with zero.",
        "2. We select domestic-bank classes N, NM, SM, SB, SI, SL on forms 31, 41, 51. This eligibility rule removes these reports before modeling.",
        "3. We do not borrow a domestic-bank median to invent equity for a different reporting group.",
        f"4. STALP has {post_conversion.STALP.isna().sum():,} blanks. State is not one of the five inputs, so we leave those labels missing.",
        "5. The later median imputer is fitted only on training features. It is a safeguard, not the treatment used for these source EQ gaps.",
    ],
)

In [ ]:
source_summary = post_conversion[["DEPDOM", "ASSET", "LNLSNET", "CHBAL", "EQ"]].describe()
display(source_summary)

In [ ]:
READING_GUIDES['reporting_banks_time'] = 'The horizontal axis is the reporting quarter; height counts banks with a source report. A falling count changes the population represented by later summaries. It does not identify why individual banks disappear.'
READING_GUIDES['median_deposits_time'] = 'The horizontal axis is report date. Height is the middle reporting bank’s deposits in USD millions, not the total banking system. Changing membership can also change this median.'
display_data_chips(
    post_conversion,
    theme=theme,
    identifier_columns=["CERT"],
    datetime_columns=["REPDTE"],
    categorical_columns=["STALP", "NAME"],
    group_label="What each source column means",
)
display_cols_by_dtype(
    post_conversion.dtypes,
    theme=theme,
    group_label="How pandas stores these columns",
)
wm_render_styler(
    style_describe_wm(source_summary, theme),
    theme=theme,
    title="Source balances: the familiar describe() summary",
)

# A bank-quarter panel has a history. Count reporting banks and track the
# median deposit balance so changes in the population are visible in time.
quarterly_panel = (
    post_conversion.assign(Date=pd.to_datetime(post_conversion["REPDTE"].astype(str)))
    .groupby("Date", as_index=False)
    .agg(
        Banks=("CERT", "nunique"),
        Median_deposits=("DEPDOM", "median"),
    )
    .sort_values("Date")
)
quarterly_panel["Median deposits (USD million)"] = quarterly_panel["Median_deposits"] / 1000

fig = px.line(quarterly_panel, x="Date", y="Banks", markers=True)
fig.update_traces(line=dict(color="#0B6F75", width=3), marker=dict(size=5))
fig.update_yaxes(title="Reporting banks", rangemode="tozero")
chart(
    fig,
    "reporting_banks_time",
    "The reporting population shrinks across 2013–2024",
    "2013–2024; distinct bank certificates",
)

fig = px.line(
    quarterly_panel,
    x="Date",
    y="Median deposits (USD million)",
    markers=True,
)
fig.update_traces(line=dict(color="#A86223", width=3), marker=dict(size=5))
fig.update_yaxes(title="Median deposits (USD million)", rangemode="tozero")
chart(
    fig,
    "median_deposits_time",
    "Median reported deposits rise as the bank population changes",
    "Median domestic deposits per reporting bank; USD million",
)

assert not long_history.duplicated(keys).any()
assert not post_conversion.duplicated(keys).any()
assert not reporting.duplicated(keys).any()

# The bundled extracts must reproduce the same rows in the long audit file.
source_columns = list(recent_extract.columns)

In [ ]:
recent_rows_match = ordered(recent_extract).equals(
    ordered(
        long_history.loc[
            long_history["REPDTE"].ge(20200331),
            source_columns,
        ]
    )
)
post_conversion_rows_match = ordered(post_conversion).equals(
    ordered(
        long_history.loc[
            long_history["REPDTE"].ge(20130331),
            source_columns,
        ]
    )
)

assert recent_rows_match
assert post_conversion_rows_match
assert long_history["REPDTE"].nunique() == 60
assert post_conversion["REPDTE"].nunique() == 48


gate = {
    "all_60_audit_quarters": True,
    "duplicate_keys": 0,
    "recent_extract_exact": recent_rows_match,
    "post_2012_extract_exact": post_conversion_rows_match,
    "historical_form_crosswalk_verified": False,
    "selected_start": 2013,
    "reason": (
        "The 2013–2024 extract begins after the documented 2012 conversion and "
        "matches the long extract exactly. Pre-2013 rows remain in the audit only."
    ),
}
(OUT / "comparability_gate.json").write_text(json.dumps(gate, indent=2))

raw = post_conversion.copy()
source_path = ROOT / "data/fdic_financials_2013_2024.csv"
source_hash = hashlib.sha256(source_path.read_bytes()).hexdigest()
raw_fingerprint = pd.util.hash_pandas_object(raw, index=True).sum()

takeaway(
    "Use 2013–2024 for the primary experiment",
    (
        "The 48 post-conversion quarters match the long extract exactly. Reports "
        "from 2010–2012 stay in the audit because the field-level crosswalk "
        "remains unresolved."
    ),
)

## 2 · Which bank-quarters have an observable outcome?

**In plain terms:** **A historical forecast needs a before, a now, and an after.** Imagine June is the report we use. March supplies previous growth; September supplies the answer we later score. A gap in either neighboring report prevents this particular historical example from being constructed.

**Does a missing September answer stop us forecasting in June?** We could still issue a forecast using information available in June. The September requirement belongs to this historical evaluation population. For a future trial, record scorable banks first and track missing outcomes later. The exclusion receipt counts what happened; disappearance alone supplies no merger or failure diagnosis.



In [ ]:
# Decide which bank-quarters can receive a measured outcome.
question_card(
    title="Which bank-quarters have a real next-quarter answer?",
    theme=theme,
    body=(
        "Trace each certificate through adjacent quarters, then count every row "
        "removed by the eligibility rules."
    ),
    kicker="Forecast population",
    chip_text="QUESTION",
)


def add_adjacent_reports(frame):
    """Attach prior and next-quarter values within each FDIC certificate."""
    result = frame.sort_values(["CERT", "REPDTE"]).copy()
    result["date"] = pd.to_datetime(result["REPDTE"].astype(str))
    result["quarter_number"] = result["date"].dt.to_period("Q").astype("int64")

    grouped = result.groupby("CERT", sort=False)
    shifts = {
        "prior_deposits": ("DEPDOM", 1),
        "next_deposits": ("DEPDOM", -1),
        "prior_quarter": ("quarter_number", 1),
        "next_quarter": ("quarter_number", -1),
        "target_date": ("date", -1),
        "next_name": ("NAME", -1),
        "next_event": ("ACTEVT", -1),
    }

    for new_column, (source_column, periods) in shifts.items():
        result[new_column] = grouped[source_column].shift(periods)

    return result


def label_next_report(frame):
    """Separate ordinary missing outcomes from the end of the dataset."""
    last_quarter = frame["quarter_number"].max()

    return np.select(
        [
            frame["quarter_number"].eq(last_quarter),
            frame["next_quarter"].isna(),
            frame["next_quarter"].sub(frame["quarter_number"]).ne(1),
        ],
        [
            "Dataset end",
            "Institution stops before dataset end",
            "Gap before later report",
        ],
        default="Adjacent next report",
    )

In [ ]:
def apply_rules(frame, rules):
    """Apply eligibility rules one at a time and record their effect."""
    keep = pd.Series(True, index=frame.index)
    ledger = [
        {
            "Rule": "All source rows",
            "Remaining": len(frame),
            "Removed": 0,
        }
    ]

    for rule_name, rule_mask in rules.items():
        rows_before = int(keep.sum())
        keep &= rule_mask
        rows_after = int(keep.sum())

        ledger.append(
            {
                "Rule": rule_name,
                "Remaining": rows_after,
                "Removed": rows_before - rows_after,
            }
        )

    return frame.loc[keep].copy(), pd.DataFrame(ledger)


def add_model_fields(frame):
    """Create the three candidate targets and five current-quarter inputs."""
    result = frame.copy()

    result["growth"] = result["next_deposits"].div(result["DEPDOM"]).sub(1)
    result["log_growth"] = np.log(result["next_deposits"].div(result["DEPDOM"]))
    result["dollar_change_m"] = result["next_deposits"].sub(result["DEPDOM"]).div(1000)

    result["log_deposits"] = np.log(result["DEPDOM"])
    result["cash_ratio"] = result["CHBAL"].div(result["ASSET"])
    result["loan_ratio"] = result["LNLSNET"].div(result["ASSET"])
    result["equity_ratio"] = result["EQ"].div(result["ASSET"])
    result["prior_growth"] = np.log(result["DEPDOM"].div(result["prior_deposits"]))

    return result

In [ ]:
READING_GUIDES['next_report'] = 'Each category describes the observable next-report situation. A dataset boundary and an institution’s final observed report are different facts. Neither category alone supplies an economic cause.'
# Merge reporting metadata before building the bank timeline.
panel = raw.merge(
    reporting,
    on=["CERT", "REPDTE"],
    how="left",
    validate="one_to_one",
    indicator=True,
)
assert panel["_merge"].eq("both").all()

panel = add_adjacent_reports(panel)
panel["Next report status"] = label_next_report(panel)

# Count every reason that a future balance is unavailable.
missing_next = panel.groupby("Next report status").size().rename("Rows").reset_index()

next_report_order = [
    "Adjacent next report",
    "Dataset end",
    "Institution stops before dataset end",
    "Gap before later report",
]
missing_next = ordered_rows(
    missing_next,
    ["Next report status"],
    category_orders={"Next report status": next_report_order},
)

table(missing_next, "A missing future balance stays missing")

fig = px.bar(
    missing_next.loc[missing_next["Next report status"].ne("Adjacent next report")],
    x="Rows",
    y="Next report status",
    orientation="h",
    text="Rows",
    color_discrete_sequence=["#0B6F75"],
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(margin=dict(l=230, r=100, t=110, b=65))
chart(
    fig,
    "next_report",
    "The dataset ends before some banks have a next report",
    "Counts exclude the adjacent reports shown in the table",
)

unobserved_columns = [
    "CERT",
    "REPDTE",
    "NAME",
    "date",
    "ACTEVT",
    "Next report status",
]
panel.loc[
    panel["Next report status"].ne("Adjacent next report"),
    unobserved_columns,
].to_csv(OUT / "unobserved_outcomes.csv", index=False)

In [ ]:
# The log-growth experiment requires positive balances in all three quarters.
domestic_classes = ["N", "NM", "SM", "SB", "SI", "SL"]
comparable_forms = [31, 41, 51]

conditions = {
    "Insured domestic bank; comparable Call Report": (
        panel["BKCLASS"].isin(domestic_classes) & panel["CALLFORM"].isin(comparable_forms)
    ),
    "Adjacent prior quarter": (panel["quarter_number"].sub(panel["prior_quarter"]).eq(1)),
    "Adjacent next quarter": (panel["next_quarter"].sub(panel["quarter_number"]).eq(1)),
    "Positive deposits in prior, current, and next quarter": (
        panel["prior_deposits"].gt(0) & panel["DEPDOM"].gt(0) & panel["next_deposits"].gt(0)
    ),
    "Positive current assets": panel["ASSET"].gt(0),
}

rows, eligibility_ledger = apply_rules(panel, conditions)
rows = add_model_fields(rows)

eligibility_ledger = ordered_rows(
    eligibility_ledger,
    ["Rule"],
    category_orders={"Rule": ["All source rows", *conditions.keys()]},
)

assert np.isfinite(rows["growth"]).all()
assert np.isfinite(rows["log_growth"]).all()
assert np.isfinite(rows["prior_growth"]).all()
assert rows["growth"].ge(-1).all()
assert len(panel) - len(rows) == eligibility_ledger["Removed"].sum()

table(eligibility_ledger, "Every exclusion has a count")

takeaway(
    "The forecasting rows have three consecutive positive balances",
    (
        f"{len(rows):,} eligible bank-quarters remain. Small deposit bases stay "
        "in the sample, so the target comparison must show how strongly they "
        "stretch ordinary percentage growth."
    ),
)

In [ ]:
# The one-quarter gaps keep each split's outcomes behind the next split's inputs.
train = rows.loc[rows["date"].le("2022-09-30")].copy()
valid = rows.loc[rows["date"].between("2023-01-01", "2023-09-30")].copy()
holdout_mask = rows["date"].between("2024-01-01", "2024-09-30")
holdout = rows.loc[holdout_mask].copy()

## 3 · Do deposit declines repeat over time?

**In plain terms:** **The time plots ask whether deposit declines recur at similar points in the year.** Imagine March reports repeatedly precede more declines than December reports. A model might benefit from knowing the calendar, so we first examine whether that pattern recurs across individual years.

Read left to right through training dates. In the Q1–Q4 view, compare the separate year markers as well as the summary. A high rate means more bank-quarter examples had a negative next-quarter change. The seasonal forecast later in the notebook is a follow-up experiment developed after inspecting 2024.



In [ ]:
READING_GUIDES['decline_share_time'] = 'Each point is a training predictor quarter. Height is the share of eligible bank-quarter examples whose deposits decline next quarter. Compare the recurring rhythm with periods that differ.'
READING_GUIDES['training_seasonality'] = 'Q1–Q4 refer to predictor quarters. Each year has separate markers; the summary describes their pattern. Compare years before treating a quarter-of-year difference as stable.'
# A time line tests whether one period drives the overall decline rate.
quarterly_declines = (
    train.assign(Decline=train["growth"].lt(0))
    .groupby("date", as_index=False)
    .agg(Rows=("CERT", "size"), Declines=("Decline", "sum"))
    .sort_values("date")
)
quarterly_declines["Decline share"] = quarterly_declines["Declines"] / quarterly_declines["Rows"]
fig = px.line(
    quarterly_declines,
    x="date",
    y="Decline share",
    markers=True,
)
fig.update_traces(line=dict(color="#A86223", width=3), marker=dict(size=5))
fig.update_yaxes(title="Bank-quarters with deposit decline", tickformat=".0%")
fig.update_xaxes(title="Predictor quarter")
chart(
    fig,
    "decline_share_time",
    "Deposit declines became much less common in early 2020",
    "Training period only; next-quarter decline divided by eligible rows",
)

# Keep each year visible when checking whether the quarterly rhythm repeats.
seasonality = quarterly_declines.copy()
seasonality["Year"] = seasonality["date"].dt.year
seasonality["Quarter"] = "Q" + seasonality["date"].dt.quarter.astype(str)
# A small fixed offset exposes individual years that would otherwise overlap.
seasonality["Quarter position"] = (
    seasonality["date"].dt.quarter + (seasonality["Year"] - seasonality["Year"].mean()) * 0.025
)
seasonality.to_csv(OUT / "training_seasonality.csv", index=False)

fig = px.scatter(
    seasonality,
    x="Quarter position",
    y="Decline share",
    color="Year",
    hover_data=["Quarter", "date", "Rows", "Declines"],
    color_continuous_scale="Teal",
)
fig.update_traces(marker={"size": 10, "opacity": 0.8})
fig.update_yaxes(title="Bank-quarters with deposit decline", tickformat=".0%")
fig.update_xaxes(
    title="Predictor quarter of year",
    tickvals=[1, 2, 3, 4],
    ticktext=["Q1", "Q2", "Q3", "Q4"],
    range=[0.7, 4.3],
)
chart(
    fig,
    "training_seasonality",
    "Declines were more common after Q1; individual years varied",
    "Each dot is one training year; descriptive check only",
)
seasonal_medians = seasonality.groupby("Quarter")["Decline share"].median()
takeaway(
    "The usual Q1–Q4 rhythm breaks in 2020",
    f"The training-year median decline rate is {seasonal_medians['Q1']:.1%} "
    f"after Q1 reports and {seasonal_medians['Q4']:.1%} after Q4 reports. "
    "The 2020 points show why this remains descriptive. Quarter of year is not a model input.",
)

## 4 · What do the five inputs look like?

**In plain terms:** **The models receive five clues about each bank.** Imagine assets of USD 100 and cash of USD 15: cash/assets is 15%. Repeat that division for loans and equity. Deposit size and previous deposit growth complete the five inputs.

**Would one ratio cleanly separate banks whose deposits later fall?** Compare the two cash-ratio boxes. Their middle halves overlap substantially. Next, inspect every input's distribution: the median shows its center, the spread shows how different banks can be, and the tails show the unusual reports a mean can conceal.

**Are these five copies of the same clue?** The correlation plot measures pairwise rank relationships. Modest correlations suggest limited pairwise repetition; they do not establish independence or rule out nonlinear relationships. The prior-versus-next plot then asks whether simply repeating the previous change has useful signal.



In [ ]:
# Give each input a name, calculation, and reporting date.
question_card(
    title="What do the five inputs look like before preprocessing?",
    theme=theme,
    body=(
        "Inspect their ranges, correlations, and the relationship between prior "
        "and next-quarter growth using training rows only."
    ),
    kicker="Model inputs",
    chip_text="QUESTION",
)

# Begin with a plain-language map from FDIC fields to model inputs.
feature_ledger = pd.DataFrame(
    {
        "Input": FEATURES,
        "Calculation": [
            "ln(DEPDOM)",
            "CHBAL / ASSET",
            "LNLSNET / ASSET",
            "EQ / ASSET",
            "ln(DEPDOM / prior DEPDOM)",
        ],
        "Timing": [
            "Current quarter",
            "Current quarter",
            "Current quarter",
            "Current quarter",
            "Current and adjacent prior quarter",
        ],
    }
)
table(feature_ledger, "Five inputs; no future balance")

# Describe every feature before imputation or scaling changes its units.
feature_summary = (
    train[FEATURES]
    .describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99])
    .T.reset_index(names="Feature")
)
feature_summary = feature_summary[["Feature", "count", "mean", "1%", "50%", "99%", "max"]]
feature_summary["Feature"] = feature_summary["Feature"].map(
    {
        "log_deposits": "Bank size",
        "cash_ratio": "Cash / assets",
        "loan_ratio": "Loans / assets",
        "equity_ratio": "Equity / assets",
        "prior_growth": "Prior growth",
    }
)
table(
    feature_summary,
    "What do the five training inputs look like?",
    {"count": "{:,.0f}", **{column: "{:,.4f}" for column in ["mean", "1%", "50%", "99%", "max"]}},
    wrap_columns={"Feature": 180},
)

# The notebook's original WM profile rail is the visual companion to describe().
# Each card keeps missingness and skew beside the field's typical value.
wm_render_micro_profile_cards(
    train[FEATURES],
    theme=theme,
    columns=FEATURES,
    visible_cards=5,
    max_cards=5,
    skew_threshold=1.0,
)

In [ ]:
READING_GUIDES['cash_by_outcome_box'] = 'Compare cash/assets before the next outcome. The line inside each box is the median; the box covers the middle half. The groups overlap. The displayed window is zoomed; the quartiles use all training rows.'
# Compare cash ratios by what actually happened next quarter. Both groups
# remain in the training data; a difference here is an association.
cash_groups = train[["cash_ratio", "growth"]].copy()
cash_groups["Next quarter"] = np.where(
    cash_groups["growth"].lt(0),
    "Deposits fell",
    "No decline",
)
cash_group_summary = (
    cash_groups.groupby("Next quarter", sort=False)["cash_ratio"]
    .agg(
        Rows="count", Median="median", Q1=lambda s: s.quantile(0.25), Q3=lambda s: s.quantile(0.75)
    )
    .reset_index()
)
cash_comparison = wm_compare_fields(
    cash_groups.drop(columns="growth"),
    fields=["cash_ratio", "Next quarter"],
    kind="numeric_by_category",
)
cash_comparison.figure.update_traces(boxpoints=False)
cash_comparison.figure.update_layout(showlegend=False)
cash_window = float(train["cash_ratio"].quantile(0.99))
cash_comparison.figure.update_xaxes(
    title="Cash / assets",
    tickformat=".0%",
    range=[0, cash_window],
)
chart(
    cash_comparison.figure,
    "cash_by_outcome_box",
    "Banks with later declines held a higher median cash ratio",
    f"Training rows; view ends at the 99th percentile ({cash_window:.1%}); boxes use all rows",
)
table(
    cash_group_summary,
    "Cash-ratio quartiles · exact values",
    {"Median": "{:.1%}", "Q1": "{:.1%}", "Q3": "{:.1%}"},
)

In [ ]:
READING_GUIDES['feature_correlations'] = 'Each square compares two training inputs using Spearman rank correlation. Values near zero show little monotonic pairwise association, not proof of independence. Color strength shows relationship size.'
READING_GUIDES['persistence_eda'] = 'Compare previous growth on the horizontal axis with next growth vertically. The training points show how far the next change can depart from simply repeating the last one. Read the displayed window before judging the tails.'
# Correlation answers whether the five fixed inputs repeat the same information.
friendly_names = [
    "Bank size",
    "Cash / assets",
    "Loans / assets",
    "Equity / assets",
    "Prior deposit growth",
]
correlations = train[FEATURES].corr(method="spearman")
lower_triangle = correlations.mask(np.triu(np.ones_like(correlations, dtype=bool)))
heat_text = np.where(
    lower_triangle.notna(),
    lower_triangle.round(2).astype(str),
    "",
)

fig = go.Figure(
    go.Heatmap(
        z=lower_triangle.to_numpy(),
        x=friendly_names,
        y=friendly_names,
        zmin=-1,
        zmax=1,
        colorscale=[
            [0, "#A86223"],
            [0.5, "#F4F6F7"],
            [1, "#0B6F75"],
        ],
        text=heat_text,
        texttemplate="%{text}",
        hovertemplate=("%{y} vs %{x}<br>Spearman %{z:.3f}<extra></extra>"),
        colorbar_title="Spearman",
    )
)
fig.update_yaxes(autorange="reversed")
chart(
    fig,
    "feature_correlations",
    "Pairwise rank correlations are modest across all five inputs",
    "Training rows only; lower triangle; Spearman correlation",
)

# A display-only zoom keeps small-balance jumps from flattening the scatter.
sample_source = train[["prior_growth", "log_growth"]].dropna()
sample = sample_source.sample(
    n=min(5000, len(sample_source)),
    random_state=SEED,
)

x_lower, x_upper = sample_source["prior_growth"].quantile([0.01, 0.99])
y_lower, y_upper = sample_source["log_growth"].quantile([0.01, 0.99])
visible = sample["prior_growth"].between(x_lower, x_upper) & sample["log_growth"].between(
    y_lower, y_upper
)

fig = px.scatter(
    sample.loc[visible],
    x="prior_growth",
    y="log_growth",
    opacity=0.20,
    color_discrete_sequence=["#0B6F75"],
)
fig.add_hline(y=0, line_color="#627381")
fig.add_vline(x=0, line_color="#627381")
fig.update_xaxes(title="Prior-quarter log growth")
fig.update_yaxes(title="Next-quarter log growth")
chart(
    fig,
    "persistence_eda",
    "Similar prior growth leads to widely different next-quarter growth",
    (
        "Random training sample; middle 98% chart window; "
        f"{len(sample) - visible.sum():,} sampled points outside the view"
    ),
)

## 5 · Why revisit the original dollar target?

**In plain terms:** **Counting dollars gives large banks enormous influence.** Imagine one bank loses 1% of USD 100 billion and another loses 10% of USD 100 million. The first contributes USD 1 billion to a dollar-loss score; the second contributes USD 10 million.

The archived experiment's size-only rule nearly matched the network's dollar capture. That motivates asking about proportional growth. We retain the original calculation so readers can see why the question changed.



In [ ]:
legacy = json.loads((ROOT / "experiments/experiment_0/outputs/run_summary.json").read_text())
wm_counterintuitive_card(
    title="What a novice might overlook",
    theme=theme,
    why_misread="Capturing about 85% of decline dollars sounds impressive.",
    ordinary_process=(
        f"Ranking by bank size alone captured {100 * legacy['size_capture']:.2f}% "
        "at the same review capacity."
    ),
    conclusion_boundary=(
        f"The original network added {legacy['network_difference_pp']:.2f} "
        "percentage points. The dollar weighting made size a strong baseline."
    ),
    kicker="Original experiment",
    chip_text="LOOK TWICE",
)

## 6 · What should the model predict?

**In plain terms:** **Start with USD 100 million in deposits. Next quarter there is USD 90 million.** The dollar change is minus USD 10 million. Divide by the starting USD 100 million: ordinary growth is minus 10%.

**How can the same calculation produce 522,646%?** The real training example starts at USD 1.15 million and ends near USD 6.01 billion. The small starting balance makes the ratio enormous. Its arithmetic is valid; the economic cause needs separate evidence.

The model learns **y = log(next deposits / current deposits)**. Here, log means the natural logarithm. For 100 to 90, log(0.9) is about −0.105. To interpret a prediction, use **100 × (exp(y_hat) − 1)**. Exp reverses the logarithm; subtracting one gives growth; multiplying by 100 expresses it as a percentage. We score the resulting forecasts in percentage points. Back-transforming a mean log forecast generally differs from estimating mean ordinary growth.



In [ ]:
# Work one small change by hand before comparing training targets.
question_card(
    title="What should one prediction mean?",
    theme=theme,
    body=(
        "Compare dollars, ordinary percentage growth, and log growth on training "
        "rows before choosing the target."
    ),
    kicker="Target choice",
    chip_text="QUESTION",
)

wm_formula_card(
    title="From a balance to a change",
    theme=theme,
    items=[
        {
            "label": "Signed growth",
            "fallback": "(90 million − 100 million) / 100 million = −0.10 = −10%",
        },
        {
            "label": "Log growth",
            "fallback": "ln(90 / 100) ≈ −0.1054",
        },
    ],
)


def summarize_target(series, label, multiplier=1):
    """Return a compact full-range summary without trimming observations."""
    values = series.dropna().mul(multiplier)
    quantiles = values.quantile([0, 0.01, 0.50, 0.99, 1.00])

    return {
        "Target": label,
        "Defined rows": len(values),
        "Undefined rows": len(series) - len(values),
        "Minimum": quantiles.loc[0.00],
        "1st percentile": quantiles.loc[0.01],
        "Median": quantiles.loc[0.50],
        "99th percentile": quantiles.loc[0.99],
        "Maximum": quantiles.loc[1.00],
    }


def central_target_view(values):
    """Return the untouched values inside a display-only 1st–99th-percentile window."""
    low, high = values.quantile([0.01, 0.99])
    return values.loc[values.between(low, high)]

In [ ]:
READING_GUIDES['target_growth_side_by_side'] = 'Both panels use the same training bank-quarter population. Height counts observations. Each horizontal scale shows its own central 98% window; the maximum labels retain the full-range extremes.'
# All three summaries use only training rows. The model choice comes later.
target_specs = [
    ("Dollar change (USD million)", "dollar_change_m", 1),
    ("Signed growth (%)", "growth", 100),
    ("Log growth", "log_growth", 1),
]

target_summary = []
for target_label, column, multiplier in target_specs:
    target_summary.append(
        summarize_target(
            train[column],
            target_label,
            multiplier=multiplier,
        )
    )

ordinary_percent = 100 * train["growth"]
log_values = train["log_growth"]
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        f"Ordinary growth · max {ordinary_percent.max():,.0f}%",
        f"Log growth · max {log_values.max():.2f}",
    ],
    horizontal_spacing=0.16,
)
for column, values, color in [
    (1, ordinary_percent, "#A86223"),
    (2, log_values, "#0B6F75"),
]:
    central = central_target_view(values)
    fig.add_trace(
        go.Histogram(x=central, nbinsx=42, marker_color=color, showlegend=False),
        row=1,
        col=column,
    )
fig.update_xaxes(title="Next-quarter growth (%)", row=1, col=1)
fig.update_xaxes(title="Log growth", row=1, col=2)
fig.update_yaxes(title="Bank-quarter count", row=1, col=1)
fig.update_layout(bargap=0.02)
chart(
    fig,
    "target_growth_side_by_side",
    "One change, two very different numerical scales",
    "Training rows; each panel shows its 1st–99th-percentile window; maxima use the full data",
    height=570,
)

In [ ]:
READING_GUIDES['small_denominator_growth'] = 'Move right for larger starting deposits and up for larger positive percentage growth. Both axes are logarithmic. Labels identify the largest cases; the scatter explains the ratio, while institutional causes need separate evidence.'
# Tiny starting balances create enormous ordinary percentages. Sample the
# background for display speed, then include and label every top-five case.
positive_growth = train.loc[train["growth"].gt(0)].copy()
largest_growth = positive_growth.nlargest(5, "growth")
background = positive_growth.drop(index=largest_growth.index).sample(
    n=min(8000, len(positive_growth) - len(largest_growth)),
    random_state=SEED,
)
fig = go.Figure()
fig.add_trace(
    go.Scattergl(
        x=background["DEPDOM"] / 1000,
        y=100 * background["growth"],
        mode="markers",
        name="Other positive-growth rows",
        marker=dict(color="#A5B5BC", size=5, opacity=0.22),
        hovertemplate="Starting deposits: $%{x:,.2f}M<br>Growth: %{y:,.1f}%<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=largest_growth["DEPDOM"] / 1000,
        y=100 * largest_growth["growth"],
        mode="markers",
        text=largest_growth["NAME"].str.title(),
        name="Five largest percentages",
        marker=dict(color="#A86223", size=11),
        hovertemplate="%{text}<br>Starting deposits: $%{x:,.2f}M<br>Growth: %{y:,.1f}%<extra></extra>",
    )
)

# Leader lines separate five unusually close labels without hiding the dots.
short_names = [
    "Schwab Signature",
    "Ameriprise",
    "Mitsubishi UFJ",
    "Wells Fargo Financial",
    "Stifel Trust",
]
label_offsets = [(135, -55), (45, -92), (-55, -60), (165, 5), (-35, 48)]
for (_, bank), name, (x_offset, y_offset) in zip(
    largest_growth.iterrows(), short_names, label_offsets
):
    fig.add_annotation(
        # Plotly annotations use log coordinates on logarithmic axes.
        x=np.log10(bank["DEPDOM"] / 1000),
        y=np.log10(100 * bank["growth"]),
        text=name,
        ax=x_offset,
        ay=y_offset,
        arrowhead=2,
        arrowsize=0.6,
        arrowwidth=1,
        arrowcolor="#A86223",
        font={"size": 12, "color": "#6B4A1E"},
        bgcolor="rgba(255,255,255,.9)",
    )
fig.update_xaxes(type="log", title="Starting domestic deposits (USD million, log scale)")
fig.update_yaxes(type="log", title="Positive next-quarter growth (%, log scale)")
chart(
    fig,
    "small_denominator_growth",
    "Tiny starting balances magnify percentage growth",
    f"{len(background):,} sampled background rows plus the five largest; negative and zero growth remain in the experiment",
    height=640,
)

In [ ]:
# The table is an exact-value receipt after the two visual explanations.
target_summary = pd.DataFrame(target_summary)
target_summary = ordered_rows(
    target_summary,
    ["Target"],
    category_orders={
        "Target": [
            "Dollar change (USD million)",
            "Signed growth (%)",
            "Log growth",
        ],
    },
)
# A colored range chart gives the reader the distribution before its exact receipt.
range_colors = {
    "Dollar change (USD million)": "#6E8BA5",
    "Signed growth (%)": "#D17A3A",
    "Log growth": "#0B7A75",
}
fig = go.Figure()
for _, row in target_summary.iterrows():
    target = row["Target"]
    fig.add_trace(
        go.Scatter(
            x=[row["1st percentile"], row["99th percentile"]],
            y=[target, target],
            mode="lines",
            line=dict(color=range_colors[target], width=16),
            hovertemplate=(
                f"{target}<br>Middle 98% from %{{x[0]:,.3f}} to %{{x[1]:,.3f}}<extra></extra>"
            ),
            showlegend=False,
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[row["Median"]],
            y=[target],
            mode="markers",
            marker=dict(color="#172F3E", size=13, symbol="diamond"),
            hovertemplate=f"{target}<br>Median: %{{x:,.3f}}<extra></extra>",
            showlegend=False,
        )
    )
fig.update_xaxes(
    title="Each target uses its own units; the colored band is the middle 98%, the diamond is the median"
)
fig.update_yaxes(title="")
fig.update_layout(margin=dict(l=20, r=30, t=20, b=70))
chart(
    fig,
    "target_ranges",
    "How much do the candidate targets spread?",
    "The log target keeps the same rows but puts unusually large proportional jumps on a learnable scale.",
    height=430,
)

table(
    target_summary[["Target", "1st percentile", "Median", "99th percentile", "Maximum"]],
    "Exact target ranges used for the training decision",
    {
        column: "{:,.4f}"
        for column in ["Minimum", "1st percentile", "Median", "99th percentile", "Maximum"]
    },
)

In [ ]:
target_summary.to_csv(OUT / "target_comparison.csv", index=False)

wm_counterintuitive_card(
    title="What a novice might overlook",
    theme=theme,
    why_misread=(
        "Percentage growth sounds fair because every change is divided by the "
        "bank’s own starting balance."
    ),
    ordinary_process=(
        "A very small starting balance can turn an ordinary dollar increase into "
        "a percentage change of thousands of percent. The training maximum exceeds "
        f"{train['growth'].max():,.0%}."
    ),
    conclusion_boundary=(
        "Log growth keeps the proportional interpretation while compressing those "
        "multiplicative jumps. The model will predict log growth, and evaluation "
        "will convert predictions back to ordinary percentage points."
    ),
    kicker="Target interpretation",
    chip_text="LOOK TWICE",
)

In [ ]:
# Preserve the largest changes for audit. Extreme does not mean erroneous.
extreme_columns = [
    "CERT",
    "NAME",
    "date",
    "target_date",
    "DEPDOM",
    "next_deposits",
    "growth",
    "next_name",
    "next_event",
]
extreme_index = train["growth"].abs().nlargest(6).index
extremes = train.loc[extreme_index, extreme_columns].copy()
extremes["Name changed"] = extremes["NAME"].ne(extremes["next_name"])

shown_extremes = extremes[
    [
        "CERT",
        "NAME",
        "date",
        "DEPDOM",
        "next_deposits",
        "growth",
        "Name changed",
    ]
]
shown_extremes = (
    shown_extremes.assign(_absolute_growth=shown_extremes["growth"].abs())
    .sort_values(
        ["_absolute_growth", "CERT"],
        ascending=[False, True],
    )
    .drop(columns="_absolute_growth")
)
shown_extremes = shown_extremes.drop(columns=["CERT", "Name changed"]).rename(
    columns={
        "NAME": "Bank",
        "date": "Report date",
        "DEPDOM": "Start (USD k)",
        "next_deposits": "Next (USD k)",
        "growth": "Growth",
    }
)
# The connected dots make the scale change visible before the audit table names it.
plot_extremes = shown_extremes.sort_values(
    "Growth", key=lambda values: values.abs(), ascending=True
).copy()
fig = go.Figure()

In [ ]:
for _, row in plot_extremes.iterrows():
    label = (
        f"{row['Bank']} ({row['Report date']:%Y Q%q})"
        if hasattr(row["Report date"], "quarter")
        else row["Bank"]
    )
    fig.add_trace(
        go.Scatter(
            x=[row["Start (USD k)"] / 1000, row["Next (USD k)"] / 1000],
            y=[row["Bank"], row["Bank"]],
            mode="lines+markers",
            line=dict(color="#D17A3A", width=4),
            marker=dict(size=[10, 15], color=["#6E8BA5", "#D17A3A"]),
            hovertemplate=(
                f"{row['Bank']}<br>Start: %{{x[0]:,.1f}} USD million"
                f"<br>Next quarter: %{{x[1]:,.1f}} USD million"
                f"<br>Growth: {row['Growth']:+.1%}<extra></extra>"
            ),
            showlegend=False,
        )
    )
fig.update_xaxes(type="log", title="Domestic deposits, USD million, log scale")
fig.update_yaxes(title="Bank-quarter, ordered by absolute growth")
fig.update_layout(margin=dict(l=20, r=30, t=20, b=65))
chart(
    fig,
    "largest_training_changes",
    "What do the six largest percentage changes look like in dollars?",
    "Blue is the starting report. Orange is the next quarter. The log axis keeps the small starts and large next balances visible together.",
    height=500,
)

table(
    shown_extremes,
    "Exact values behind the six largest percentage changes",
    {"Start (USD k)": "{:,.0f}", "Next (USD k)": "{:,.0f}", "Growth": "{:+.2%}"},
    wrap_columns={"Bank": 280},
)

extremes.to_csv(OUT / "extreme_review.csv", index=False)
panel.loc[
    panel["CERT"].isin(extremes["CERT"]),
    ["CERT", "REPDTE", "NAME", "date", "DEPDOM", "ASSET", "ACTEVT", "CALLFORM"],
].to_csv(OUT / "extreme_bank_histories.csv", index=False)

wm_formula_card(
    title="The same bank-quarter, a learnable scale",
    theme=theme,
    items=[
        {"label": "Starting deposits", "fallback": "USD 1.15 million"},
        {"label": "Next quarter", "fallback": "USD 6.01 billion"},
        {"label": "Ordinary growth", "fallback": "(6,011,577 − 1,150) / 1,150 = 522,645.83%"},
        {"label": "Training target", "fallback": "y = ln(D next / D now) = 8.56"},
        {
            "label": "Interpret a forecast",
            "fallback": "ordinary growth (%) = 100 × (exp(predicted y) − 1)",
        },
    ],
)

TARGET = "log_growth"

In [ ]:
target_decision = {
    "target": TARGET,
    "model_units": "log change",
    "reported_error_units": "ordinary percentage points",
    "reason": (
        "The 2013–2022 training distribution contains percentage growth above "
        "5,000x because some starting balances are tiny. Log growth preserves "
        "multiplicative movement and keeps those rows in the experiment."
    ),
    "clipping": None,
    "extreme_treatment": "Retain and audit the surrounding reports.",
    "training_only": True,
}
(OUT / "target_decision.json").write_text(json.dumps(target_decision, indent=2))

takeaway(
    "Train on log growth; translate forecasts back to percentage growth",
    (
        "The decision comes from the training distribution and the meaning of the "
        "target. No row with valid positive current and next-quarter deposits "
        "is removed simply because its growth is extreme. A same-name report "
        "does not establish why a bank’s balance changed."
    ),
)

## 7 · Did a future outcome enter training?

**In plain terms:** **A December prediction needs the following March to reveal its answer.** If we train through December 2022 predictors, those answers would spill into 2023. We stop training predictors in September 2022, whose December outcomes remain inside the training stage. The December 2023 predictor is omitted for the same boundary reason.

The timeline separates 38 training predictor quarters from three validation and three historical evaluation quarters. Earlier reports generally also have answers. Those answers serve learning or validation. The 2024 outcomes have been examined repeatedly during development, so the evaluation is reused.



In [ ]:
READING_GUIDES['split'] = 'Predictor dates provide inputs; following-quarter dates provide answers. Read the gaps between stages and the number of predictor quarters. The reused 2024 rows were excluded from fitting the original models.'
# Check the date boundaries before fitting any model.
question_card(
    title="Has any future quarter leaked into training?",
    theme=theme,
    body=(
        "The last training outcome must occur before the first validation "
        "predictor. Apply the same check at the holdout boundary."
    ),
    kicker="Time split",
    chip_text="QUESTION",
)


def describe_split(name, frame):
    """Return one readable row for the chronological split receipt."""
    return {
        "Period": name,
        "Rows": len(frame),
        "Predictor quarters": frame["date"].nunique(),
        "Banks": frame["CERT"].nunique(),
        "First predictor": frame["date"].min(),
        "Last predictor": frame["date"].max(),
        "Last outcome": frame["target_date"].max(),
    }


assert train["target_date"].max() < valid["date"].min()
assert valid["target_date"].max() < holdout["date"].min()
assert set(FEATURES).isdisjoint(
    {
        "growth",
        "log_growth",
        "next_deposits",
        "target_date",
        "dollar_change_m",
    }
)

split_summary = pd.DataFrame(
    [
        describe_split("Training", train),
        describe_split("Validation", valid),
        describe_split("Reused holdout", holdout),
    ]
)
split_summary = ordered_rows(
    split_summary,
    ["Period"],
    category_orders={
        "Period": ["Training", "Validation", "Reused holdout"],
    },
)
table(split_summary, "Accounting dates define this retrospective experiment")

fig = go.Figure()
period_colors = {
    "Training": "#0B6F75",
    "Validation": "#A86223",
    "Reused holdout": "#3F6294",
}

In [ ]:
# Named columns keep bank counts out of the calendar axis.
for record in split_summary.to_dict("records"):
    period = record["Period"]
    dates = (
        rows.loc[rows["date"].between(record["First predictor"], record["Last predictor"]), "date"]
        .drop_duplicates()
        .sort_values()
    )
    fig.add_trace(
        go.Scatter(
            x=dates,
            y=[period] * len(dates),
            mode="lines+markers",
            name=period,
            line={"width": 7, "color": period_colors[period]},
            marker={"size": 8},
            hovertemplate="%{x|%b %Y}<br>Predictor report<extra>%{fullData.name}</extra>",
        )
    )
    fig.add_annotation(
        x=record["First predictor"],
        y=period,
        yshift=26,
        text=f"{record['Predictor quarters']} quarters · {record['Rows']:,} examples",
        showarrow=False,
        xanchor="left" if period == "Training" else "right",
    )
fig.update_xaxes(
    title="Predictor report date",
    type="date",
    dtick="M12",
    tickformat="%Y",
    range=["2013-01-01", "2025-03-01"],
)
fig.update_yaxes(categoryorder="array", categoryarray=["Reused holdout", "Validation", "Training"])
fig.update_layout(showlegend=False)
chart(
    fig,
    "split",
    "38 quarters teach the model; later quarters select and evaluate it",
    "Each dot is a predictor quarter. December boundary gaps keep outcomes in their assigned stage.",
)

split_summary.to_csv(OUT / "splits.csv", index=False)
takeaway(
    "Fit on the past; judge on later reports",
    (
        "Training rows set imputation and scaling. Validation chooses Ridge "
        "strength and training duration. The reused 2024 holdout measures the "
        "previously selected design. Later research choices were informed by viewing 2024."
    ),
)

## 8 · What does the network learn?

**In plain terms:** **Imagine a forecast is too high. Which number should the network change?** A weight controls how strongly an input affects the prediction. The small worked example calculates an error, adjusts a weight, and checks whether the error shrinks.

The full network repeats that kind of adjustment across 737 parameters. Its five inputs feed 32 hidden units, then 16, then one log-growth prediction. A successful training update tells us the optimization worked. Later observations test whether what it learned travels through time.



In [ ]:
READING_GUIDES['architecture'] = 'Follow five inputs through the 32-unit and 16-unit hidden layers to one output. That output is log growth; the later conversion expresses the forecast as ordinary percentage growth.'
# Start with one neuron before showing the full network.
question_card(
    title="What changes when a neural network learns?",
    theme=theme,
    body="Follow one weighted sum, one ReLU, and one gradient step before looking at the full 5–32–16–1 architecture.",
    kicker="Network mechanics",
    chip_text="QUESTION",
)

# Use tiny numbers first. The full network repeats this same arithmetic across
# many connected weights.
wm_formula_card(
    title="One neuron, one visible calculation",
    theme=theme,
    items=[
        {"label": "Weighted sum", "fallback": "z = 2 × 0.5 + 3 × (−0.2) + 0.1 = 0.5"},
        {"label": "ReLU", "fallback": "max(0, z) = 0.5"},
        {"label": "Squared error", "fallback": "If the answer is 0.8: (0.5 − 0.8)² = 0.09"},
    ],
)
# This hand-checkable update teaches the gradient mechanics.
x_demo = np.array([2.0, 3.0])
w_demo = np.array([0.5, -0.2])
bias_demo = 0.1
answer_demo = 0.8
pred_demo = x_demo @ w_demo + bias_demo
error_demo = pred_demo - answer_demo
w_after = w_demo - 0.01 * (2 * error_demo * x_demo)
bias_after = bias_demo - 0.01 * (2 * error_demo)
new_error = (x_demo @ w_after + bias_after - answer_demo) ** 2
assert new_error < error_demo**2
table(
    pd.DataFrame(
        {
            "Stage": ["Before one update", "After one update"],
            "Squared error": [error_demo**2, new_error],
        }
    ),
    "A gradient step we can check by hand",
    {"Squared error": "{:.6f}"},
)
fig = go.Figure(
    go.Scatter(
        x=[0, 1, 2, 3],
        y=[0, 0, 0, 0],
        mode="lines+markers+text",
        text=["5 financial inputs", "32 ReLU units", "16 ReLU units", "1 linear output"],
        textposition="top center",
        marker=dict(size=22, color="#0B6F75"),
        line=dict(color="#3F6294"),
    )
)
fig.update_xaxes(visible=False, range=[-0.5, 3.5])
fig.update_yaxes(visible=False, range=[-0.3, 0.5])
chart(
    fig,
    "architecture",
    "Five inputs become one growth forecast",
    "737 trainable weights and biases",
    height=380,
)

## 9 · How are the four forecasts fitted?

**In plain terms:** **Give every bank 0% growth. You have built the first forecast.** It has no weights to fit. Persistence copies the previous quarter's growth. Ridge learns a regularized linear relationship from the five inputs. The MLP learns weights in its two hidden layers.

**How do we decide when to stop changing a model?** Use the later validation period. Ridge chooses alpha by validation MAE in ordinary percentage points. The original MLP chooses its stopping point by validation mean squared error in log growth, restoring the best weights. These are different selection rules, and we keep that history visible. Imputation and scaling learn their parameters from training rows only.



In [ ]:
# Learn missing-value replacements and scales from training rows only.
def fit_preprocessor(training_frame):
    """Fit median imputation and standardization on training rows only."""
    fitted_imputer = SimpleImputer(strategy="median")
    fitted_scaler = StandardScaler()

    imputed = fitted_imputer.fit_transform(training_frame[FEATURES])
    fitted_scaler.fit(imputed)

    return fitted_imputer, fitted_scaler


def transform_features(frame, fitted_imputer, fitted_scaler):
    """Apply the frozen preprocessing steps to one time split."""
    imputed = fitted_imputer.transform(frame[FEATURES])
    transformed = fitted_scaler.transform(imputed)
    return transformed.astype("float32")


imputer, scaler = fit_preprocessor(train)
X_train = transform_features(train, imputer, scaler)
X_valid = transform_features(valid, imputer, scaler)
y_train = train[TARGET].to_numpy(dtype="float32")
y_valid = valid[TARGET].to_numpy(dtype="float32")

assert np.isfinite(X_train).all()
assert np.isfinite(X_valid).all()
assert np.allclose(imputer.statistics_, train[FEATURES].median())

preprocessing = pd.DataFrame(
    {
        "Feature": FEATURES,
        "Training median": imputer.statistics_,
        "Training mean after imputation": scaler.mean_,
        "Training scale": scaler.scale_,
    }
)
table(preprocessing, "Parameters learned only from training")
preprocessing.to_csv(OUT / "preprocessing.csv", index=False)

# Freeze every choice that could otherwise drift after seeing 2024.
settings = {
    "target": target_decision,
    "features": FEATURES,
    "seed": 42,
    "additional_seeds": [7, 99],
    "architecture": [5, 32, 16, 1],
    "optimizer": "Adam",
    "learning_rate": 0.001,
    "loss": "MSE on log growth",
    "batch_size": 512,
    "epochs_max": 200,
    "early_stopping_patience": 10,
    "ridge_alphas": [0.01, 0.1, 1, 10, 100],
    "ridge_selection": "validation MAE in percentage points",
    "historical_status": "reused 2024 holdout",
    "clipping": None,
    "train_predictor_end": "2022-09-30",
    "validation_predictors": ["2023-01-01", "2023-09-30"],
    "holdout_predictors": ["2024-01-01", "2024-09-30"],
    "tail_definition": ("realized bottom 25% and 10% separately within each holdout quarter"),
    "ranking": ("lowest ceil(10% of rows) per quarter; ties broken by CERT"),
    "bootstrap": (
        "500 paired bank-cluster resamples; MAE difference MLP minus Ridge; fixed trained models"
    ),
    "eligibility": list(conditions),
    "source_sha256": source_hash,
}
(OUT / "design_plan.json").write_text(json.dumps(settings, indent=2))

In [ ]:
# Tune the linear comparison on validation data.
def choose_ridge(X_fit, y_fit, X_check, y_check, alphas):
    """Select Ridge strength using validation MAE in percentage points."""
    rows = []
    models = {}

    for alpha in alphas:
        candidate = Ridge(alpha=alpha).fit(X_fit, y_fit)
        models[alpha] = candidate
        rows.append(
            {
                "Alpha": alpha,
                **log_scores(y_check, candidate.predict(X_check)),
            }
        )

    results = pd.DataFrame(rows)
    chosen_alpha = float(results.sort_values(["MAE (pp)", "Alpha"]).iloc[0]["Alpha"])
    return models[chosen_alpha], chosen_alpha, results


ridge, best_alpha, ridge_tuning = choose_ridge(
    X_train,
    y_train,
    X_valid,
    y_valid,
    settings["ridge_alphas"],
)

In [ ]:
# Build one fixed neural-network architecture, then let early stopping choose
# how long it trains.
def build_network(seed):
    """Create the fixed 5 → 32 → 16 → 1 neural network."""
    tf.keras.utils.set_random_seed(seed)

    network = tf.keras.Sequential(
        [
            tf.keras.Input(shape=(5,)),
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(16, activation="relu"),
            tf.keras.layers.Dense(1),
        ]
    )
    network.compile(
        optimizer=tf.keras.optimizers.Adam(0.001),
        loss="mse",
    )

    assert network.count_params() == 737
    return network


def fit_network(seed):
    """Train one fixed network and restore its best validation weights."""
    network = build_network(seed)

    options = tf.data.Options()
    options.threading.private_threadpool_size = 2

    training_data = (
        tf.data.Dataset.from_tensor_slices((X_train, y_train))
        .shuffle(len(y_train), seed=seed)
        .batch(512)
        .with_options(options)
    )
    validation_data = (
        tf.data.Dataset.from_tensor_slices((X_valid, y_valid)).batch(512).with_options(options)
    )

    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
    )

    started = time.perf_counter()
    fitted = network.fit(
        training_data,
        validation_data=validation_data,
        epochs=200,
        callbacks=[early_stopping],
        verbose=0,
        shuffle=False,
    )
    elapsed = time.perf_counter() - started

    return network, fitted.history, elapsed


def predict_log_growth(model, features):
    """Return a flat NumPy array from a Keras model."""
    return np.asarray(model(features, training=False)).ravel()

In [ ]:
READING_GUIDES['validation'] = 'Compare candidate Ridge settings using validation error. Smaller MAE chooses alpha. This is the settings-selection period, separate from the reused 2024 score.'
# Fit once with the frozen seed; validation decides when to stop.
mlp, history, training_seconds = fit_network(SEED)

validation_predictions = {
    "Zero growth": np.zeros(len(valid)),
    "Persistence": valid["prior_growth"].to_numpy(),
    "Ridge": ridge.predict(X_valid),
    "MLP": predict_log_growth(mlp, X_valid),
}
validation_scores = pd.DataFrame(
    [
        {
            "Model": model_name,
            **log_scores(y_valid, prediction),
        }
        for model_name, prediction in validation_predictions.items()
    ]
)
validation_scores = ordered_rows(
    validation_scores,
    ["Model"],
    category_orders={"Model": MODEL_ORDER},
)

fig = px.scatter(
    validation_scores,
    x="MAE (pp)",
    y="RMSE (pp)",
    color="Model",
    text="Model",
    color_discrete_map=MODEL_COLORS,
)
fig.update_traces(textposition="top center", marker=dict(size=15))
fig.add_vline(x=validation_scores["MAE (pp)"].median(), line_dash="dot", line_color="#8A949B")
fig.add_hline(y=validation_scores["RMSE (pp)"].median(), line_dash="dot", line_color="#8A949B")
fig.update_xaxes(title="Typical miss: MAE in percentage points, lower is better")
fig.update_yaxes(title="Large-miss penalty: RMSE in percentage points, lower is better")
fig.update_layout(showlegend=False, margin=dict(l=30, r=30, t=20, b=70))
chart(
    fig,
    "validation",
    "Which model has the smallest typical miss and fewest large misses?",
    "Each dot is one method on the same 2023 Q1–Q3 bank-quarters. The lower-left region is better on both measures.",
    height=500,
)

table(
    validation_scores.sort_values("MAE (pp)"),
    "Exact validation errors after the visual comparison",
    {"MAE (pp)": "{:.4f}", "RMSE (pp)": "{:.4f}"},
)

best_epoch = int(np.argmin(history["val_loss"]) + 1)
frozen = {
    **settings,
    "chosen_ridge_alpha": best_alpha,
    "mlp_best_epoch": best_epoch,
    "validation_winner": (validation_scores.sort_values("MAE (pp)").iloc[0]["Model"]),
    "selection_complete_before_holdout_predictions": True,
}
(OUT / "frozen_plan.json").write_text(json.dumps(frozen, indent=2))
validation_scores.to_csv(OUT / "validation_scores.csv", index=False)
ridge_tuning.to_csv(OUT / "ridge_tuning.csv", index=False)

In [ ]:
# Inspect which observed outcomes dominate validation MSE.
history_frame = pd.DataFrame(history).rename_axis("Epoch").reset_index()
history_frame["Epoch"] += 1
history_frame.to_csv(OUT / "learning_curve.csv", index=False)

validation_audit = valid[
    ["CERT", "NAME", "date", "DEPDOM", "next_deposits", "growth", "next_name"]
].copy()
validation_mlp_growth = np.expm1(validation_predictions["MLP"])
validation_audit["Squared MLP error"] = (
    validation_audit["growth"].to_numpy() - validation_mlp_growth
) ** 2
validation_audit["Share of squared error"] = (
    validation_audit["Squared MLP error"] / validation_audit["Squared MLP error"].sum()
)
validation_audit.nlargest(10, "Squared MLP error").to_csv(
    OUT / "validation_extremes.csv",
    index=False,
)

error_leaders = validation_audit.nlargest(10, "Squared MLP error").copy()
error_leaders["Label"] = (
    error_leaders["NAME"].str.title() + " · " + error_leaders["date"].dt.strftime("%Y Q%q")
)
fig = px.bar(
    error_leaders.sort_values("Share of squared error"),
    x="Share of squared error",
    y="Label",
    orientation="h",
    color="Share of squared error",
    color_continuous_scale=["#D9E8E5", "#0B7A75", "#D17A3A"],
    text="Share of squared error",
)
fig.update_traces(texttemplate="%{text:.1%}", textposition="outside", cliponaxis=False)
fig.update_xaxes(
    title="Share of total squared error",
    tickformat=".0%",
    range=[0, min(1.08, error_leaders["Share of squared error"].max() * 1.12)],
)
fig.update_yaxes(title="Bank-quarter, ordered from largest contribution")
fig.update_layout(coloraxis_showscale=False, margin=dict(l=30, r=70, t=20, b=65))
chart(
    fig,
    "validation_error_concentration",
    "Which bank-quarters drive the model’s large-error score?",
    "Each bar is one validation outcome. Squared error gives very large misses much more weight, so one bar can dominate the total.",
    height=510,
)

table(
    error_leaders.head(3)[
        ["CERT", "NAME", "date", "DEPDOM", "next_deposits", "growth", "Share of squared error"]
    ],
    "Exact values for the three largest contributors to validation squared error",
    {"growth": "{:+.1%}", "Share of squared error": "{:.1%}"},
)

worst = validation_audit.nlargest(1, "Squared MLP error").iloc[0]
takeaway(
    "One extreme outcome can dominate MSE",
    (
        f"{worst.NAME} at {worst.date:%Y-%m-%d} contributes "
        f"{worst['Share of squared error']:.1%} of validation squared error. "
        f"Its growth is {worst.growth:.1%}. The source history remains available "
        "for review, and the frozen model stays unchanged."
    ),
)

In [ ]:
READING_GUIDES['learning'] = 'Move right through training epochs. Compare training and validation loss in log-growth space. The restored checkpoint comes from the best validation epoch, not automatically the final epoch.'
# The learning curve shows the validation-selected training duration.
fig = go.Figure()
for column, label, color, dash in [
    ("loss", "Training", "#0B6F75", "solid"),
    ("val_loss", "Validation", "#A86223", "dash"),
]:
    fig.add_trace(
        go.Scatter(
            x=history_frame["Epoch"],
            y=history_frame[column],
            name=label,
            line={"color": color, "dash": dash},
        )
    )

fig.update_xaxes(title="Epoch")
fig.update_yaxes(title="Mean squared log-growth error", type="log")
chart(
    fig,
    "learning",
    "Validation loss determines when neural-network training stops",
    "Logarithmic loss axis; best validation weights restored",
)

takeaway(
    "Training duration comes from validation",
    (
        f"The MLP restored epoch {best_epoch} after {len(history_frame)} epochs. "
        f"Training took {training_seconds:.1f} seconds on this run. The 2024 "
        "period played no role in choosing the weights."
    ),
)

## 10 · Which forecast errs least?

**In plain terms:** **A forecast can make smaller average misses and still make a worse large miss.** Work the three-error example below before reading the real scores. MAE averages the absolute misses. RMSE squares each miss first, giving a large error much more influence, then takes a square root.

Among the original four forecasts, zero growth has the lowest MAE and the MLP has the lowest RMSE. The follow-up seasonal median later lowers historical MAE further. The error plots show where the original RMSE difference comes from; a metric label alone cannot tell us which individual forecasts improved.



In [ ]:
# EXEMPLAR: formula-card
# Illustration: three absolute errors, expressed in percentage points.
# Compare the average miss with a score that puts extra weight on large misses.
toy_errors = pd.DataFrame({"Forecast A": [1.0, 1.0, 7.0], "Forecast B": [4.0, 4.0, 4.0]})
toy_scores = pd.DataFrame(
    {
        "MAE (pp)": toy_errors.mean(),
        "RMSE (pp)": np.sqrt(toy_errors.pow(2).mean()),
    }
)
display(toy_errors.rename_axis("Illustrative case").round(2))
display(toy_scores.round(3))
toy_scores.to_csv(OUT / "metric_illustration.csv")
question_card(
    title="Would you prefer two tiny misses and one large miss?",
    theme=theme,
    body="Forecast A misses by 1, 1, and 7 percentage points. Forecast B misses by 4 points every time. The score you choose changes the answer.",
    kicker="Three invented forecasts",
    chip_text="TRY IT",
)
wm_counterintuitive_card(
    title="The seven-point miss changes the ordering",
    theme=theme,
    why_misread="A is closer on two of the three cases: 1 point versus 4 points.",
    ordinary_process="MAE adds the misses: A totals 9 points; B totals 12. Divide by three: A scores 3, B scores 4. A has the smaller average absolute miss.",
    conclusion_boundary="RMSE squares each miss first. A: 1 + 1 + 49 = 51. B: 16 + 16 + 16 = 48. Divide by three and take the square root: A scores 4.123, B scores 4. B now scores lower.",
    kicker="MAE versus RMSE",
    chip_text="LOOK TWICE",
)
big_number_card(
    title="One large miss can change the comparison",
    theme=theme,
    value="49 of 51",
    value_label="A’s squared-error total comes mostly from one case",
    body="The seven-point miss contributes 49 after squaring.\nThe two one-point misses contribute only 2 together.\nThese are invented examples. The next chart evaluates the real bank forecasts.",
)

In [ ]:
# Compare the four forecasts on the same later bank-quarters.
question_card(
    title="Okay, did the MLP actually reduce error on later quarters?",
    theme=theme,
    body=(
        "Score every model on the same bank-quarters. Read MAE and RMSE "
        "together because they reward different error behavior."
    ),
    kicker="Historical evaluation",
    chip_text="QUESTION",
)

# Open the reused holdout after every modeling choice has been frozen.
test = rows.loc[holdout_mask].copy()
X_test = transform_features(test, imputer, scaler)
y_test = test[TARGET].to_numpy()

log_predictions = {
    "Zero growth": np.zeros(len(test)),
    "Persistence": test["prior_growth"].to_numpy(),
    "Ridge": ridge.predict(X_test),
    "MLP": predict_log_growth(mlp, X_test),
}

for prediction in log_predictions.values():
    assert len(prediction) == len(test)
    assert np.isfinite(prediction).all()

metrics = pd.DataFrame(
    [
        {
            "Model": model_name,
            **log_scores(y_test, prediction),
        }
        for model_name, prediction in log_predictions.items()
    ]
)
metrics = ordered_rows(
    metrics,
    ["Model"],
    category_orders={"Model": MODEL_ORDER},
)

In [ ]:
READING_GUIDES['comparison'] = 'Read each panel horizontally: farther left means smaller error. MAE averages absolute percentage-point misses. RMSE puts more weight on large misses. These are the original four models on identical 2024 rows.'
# The same four models appear in both panels. A dot's position carries the
# score, while its label keeps small differences readable.
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=[
        "Average absolute miss",
        "Large-error-sensitive miss",
    ],
    horizontal_spacing=0.22,
)

for column_number, metric_name in enumerate(
    ["MAE (pp)", "RMSE (pp)"],
    start=1,
):
    metric_values = metrics.set_index("Model")[metric_name].reindex(MODEL_ORDER)
    winner_name = metric_values.idxmin()
    for model_name, value in metric_values.items():
        fig.add_trace(
            go.Scatter(
                x=[value],
                y=[model_name],
                mode="markers+text",
                marker={
                    "size": 14 if model_name == winner_name else 10,
                    "color": "#0B6F75" if model_name == winner_name else "#9AA6B2",
                },
                text=[f"{value:.3f} pp"],
                textposition="middle right",
                textfont={"color": "#26323A"},
                showlegend=False,
                hovertemplate=f"{model_name}<br>{metric_name}: {value:.4f} pp<extra></extra>",
            ),
            row=1,
            col=column_number,
        )
    fig.update_xaxes(
        title="Percentage points; lower is better",
        range=[max(0, metric_values.min() - 0.25), metric_values.max() + 0.9],
        row=1,
        col=column_number,
    )
    fig.update_yaxes(
        categoryorder="array",
        categoryarray=list(reversed(MODEL_ORDER)),
        row=1,
        col=column_number,
    )

chart(
    fig,
    "comparison",
    "Zero growth leads MAE; the neural network leads RMSE",
    "Zero growth has the lowest MAE; the MLP has the lowest RMSE among the four core models.",
    height=500,
)

In [ ]:
# Check the large misses directly before interpreting the RMSE result.
large_error_check = pd.DataFrame(
    [
        {
            "Model": model_name,
            "99th percentile absolute error (pp)": float(
                np.quantile(np.abs(100 * (np.expm1(y_test) - np.expm1(prediction))), 0.99)
            ),
        }
        for model_name, prediction in log_predictions.items()
    ]
)
large_error_check.to_csv(OUT / "large_error_quantiles.csv", index=False)
assert (
    large_error_check.loc[
        large_error_check.Model.eq("MLP"),
        "99th percentile absolute error (pp)",
    ].iloc[0]
    < large_error_check.loc[
        large_error_check.Model.eq("Zero growth"),
        "99th percentile absolute error (pp)",
    ].iloc[0]
)

In [ ]:
# EXEMPLAR: analytical-question
question_card(
    title="Wait. How can the MLP win RMSE but lose MAE?",
    theme=theme,
    body=(
        "A 10-point miss contributes 10 to absolute error but 100 to "
        "squared error. Group the same 2024 rows by the zero-growth miss "
        "and see where the MLP traded accuracy."
    ),
    kicker="Read the scorecard",
    chip_text="QUESTION",
)

In [ ]:
READING_GUIDES['error_tradeoff'] = 'Compare absolute-error quantiles on identical 2024 rows. Higher quantiles describe increasingly large misses. This helps locate the MLP’s RMSE advantage instead of assuming every forecast improved.'
# Group by realized baseline error for diagnosis only. This grouping uses
# the future outcome and cannot be used to select a bank in advance.
zero_absolute_error = np.abs(100 * (np.expm1(y_test) - np.expm1(log_predictions["Zero growth"])))
mlp_absolute_error = np.abs(100 * (np.expm1(y_test) - np.expm1(log_predictions["MLP"])))

cutoffs = np.quantile(zero_absolute_error, [0.90, 0.99])
error_bands = pd.DataFrame(
    {
        "Zero-growth miss (pp)": zero_absolute_error,
        "MLP miss (pp)": mlp_absolute_error,
    }
)
error_bands["Band"] = pd.cut(
    error_bands["Zero-growth miss (pp)"],
    bins=[-np.inf, *cutoffs, np.inf],
    labels=["Smallest 90%", "Next 9%", "Largest 1%"],
)

tradeoff = (
    error_bands.groupby("Band", observed=True)
    .agg(
        Rows=("Band", "size"),
        Zero_growth_MAE_pp=("Zero-growth miss (pp)", "mean"),
        MLP_MAE_pp=("MLP miss (pp)", "mean"),
    )
    .reset_index()
)
tradeoff["MLP minus zero (pp)"] = tradeoff["MLP_MAE_pp"] - tradeoff["Zero_growth_MAE_pp"]
tradeoff.to_csv(OUT / "error_tradeoff.csv", index=False)

fig = go.Figure()
fig.add_bar(
    x=tradeoff["MLP minus zero (pp)"],
    y=tradeoff["Band"].astype(str),
    orientation="h",
    marker_color=[
        "#A86223" if delta > 0 else "#0B6F75" for delta in tradeoff["MLP minus zero (pp)"]
    ],
    text=[f"{delta:+.2f} pp" for delta in tradeoff["MLP minus zero (pp)"]],
    textposition="outside",
    customdata=tradeoff["Rows"],
    hovertemplate=(
        "%{y}<br>MLP minus zero-growth MAE: %{x:+.3f} pp"
        "<br>%{customdata:,} bank-quarters<extra></extra>"
    ),
)
fig.add_vline(x=0, line_color="#343B43", line_width=1.5)
fig.update_xaxes(
    title="MLP minus zero-growth mean absolute miss (pp)",
    range=[-4.5, 1.2],
)
fig.update_yaxes(
    title="",
    categoryorder="array",
    categoryarray=["Largest 1%", "Next 9%", "Smallest 90%"],
)
chart(
    fig,
    "error_tradeoff",
    "The MLP helps most where zero growth misses most",
    "Teal left of zero: MLP helped. Amber right: MLP hurt. Outcome-defined groups.",
    height=480,
)

In [ ]:
common = tradeoff.loc[tradeoff.Band.eq("Smallest 90%")].iloc[0]
largest = tradeoff.loc[tradeoff.Band.eq("Largest 1%")].iloc[0]
takeaway(
    "The MLP trades small misses for fewer large misses",
    (
        f"On {int(common.Rows):,} smaller-miss rows, its average absolute "
        f"error was {common['MLP minus zero (pp)']:+.2f} pp relative to "
        f"zero growth. On the largest {int(largest.Rows):,} baseline misses, "
        f"it was {largest['MLP minus zero (pp)']:+.2f} pp. "
        "These groups use realized outcomes, so this chart explains the "
        "scores after the fact; it cannot choose banks for review beforehand."
    ),
)

rmse_so_far = metrics.set_index("Model")["RMSE (pp)"]
relative_rmse_drop = (
    100 * (rmse_so_far["Zero growth"] - rmse_so_far["MLP"]) / rmse_so_far["Zero growth"]
)
takeaway(
    "The RMSE improvement changes large-error scoring",
    (
        f"The MLP lowered RMSE by {relative_rmse_drop:.1f}% relative to zero "
        "growth, yet zero growth still had the lower MAE. This measures "
        "forecast error across many bank-quarters, not the chance that any "
        "one bank needs investigation. The next check asks whether predicted "
        "rankings put more truly weak-growth banks into a fixed-size review "
        "list. An analyst must still verify the source report and context."
    ),
)

In [ ]:
READING_GUIDES['actual_predicted'] = 'A point on the diagonal would have the correct predicted growth. Distance from it is a miss. Both axes use matching display windows; the scores still include every eligible evaluation row.'
# Convert log predictions back to ordinary growth for interpretation.
growth_predictions = {
    model_name: np.expm1(prediction) for model_name, prediction in log_predictions.items()
}

scatter = test[["CERT", "NAME", "date", "growth"]].copy()
scatter["Actual (%)"] = 100 * scatter["growth"]
scatter["Predicted (%)"] = 100 * growth_predictions["MLP"]

actual_limits = scatter["Actual (%)"].quantile([0.01, 0.99])
predicted_limits = scatter["Predicted (%)"].quantile([0.01, 0.99])
central_mask = scatter["Actual (%)"].between(*actual_limits) & scatter["Predicted (%)"].between(
    *predicted_limits
)
central_scatter = scatter.loc[central_mask].copy()

fig = px.scatter(
    central_scatter,
    x="Actual (%)",
    y="Predicted (%)",
    hover_data=["NAME", "CERT", "date"],
    opacity=0.30,
    color_discrete_sequence=["#0B6F75"],
)

lower_bound = min(
    central_scatter["Actual (%)"].min(),
    central_scatter["Predicted (%)"].min(),
)
upper_bound = max(
    central_scatter["Actual (%)"].max(),
    central_scatter["Predicted (%)"].max(),
)
span = max(upper_bound - lower_bound, 1)
bounds = [
    lower_bound - 0.04 * span,
    upper_bound + 0.04 * span,
]

fig.add_trace(
    go.Scatter(
        x=bounds,
        y=bounds,
        mode="lines",
        name="Ideal forecast",
        line={"color": "#343B43", "dash": "dash"},
    )
)
fig.update_xaxes(range=bounds)
fig.update_yaxes(
    range=bounds,
    scaleanchor="x",
    scaleratio=1,
)
chart(
    fig,
    "actual_predicted",
    "Most MLP forecasts stay close to zero growth",
    (
        f"Central 98% view; {len(scatter) - len(central_scatter):,} extreme "
        "rows remain in every score and table"
    ),
    height=700,
)

In [ ]:
# Save one row per bank-quarter with ordinary-growth predictions.
result_frame = test[
    [
        "CERT",
        "NAME",
        "date",
        "target_date",
        "growth",
        "DEPDOM",
        "next_deposits",
    ]
].reset_index(drop=True)

for model_name, prediction in growth_predictions.items():
    result_frame[model_name] = prediction

result_frame.to_csv(OUT / "predictions.csv", index=False)
metrics.to_csv(OUT / "historical_scores.csv", index=False)

mae = metrics.set_index("Model")["MAE (pp)"]
winner = mae.idxmin()
difference = float(mae["MLP"] - mae["Ridge"])

takeaway(
    "Zero growth wins MAE; the MLP wins RMSE",
    (
        f"Zero growth: {mae['Zero growth']:.3f} pp MAE. "
        f"MLP: {mae['MLP']:.3f} pp. "
        f"Ridge: {mae['Ridge']:.3f} pp. "
        f"Persistence: {mae['Persistence']:.3f} pp. "
        f"The MLP minus Ridge difference is {difference:+.3f} pp. "
        "The MLP reduces some large misses, but its typical absolute miss is "
        "larger than zero growth on this reused holdout."
    ),
    f"{mae[winner]:.3f} pp",
)

## 11 · Where are the large misses?

**In plain terms:** **Imagine two models have similar average error. Do they miss the same banks?** The actual-versus-predicted plot shows where each forecast sits relative to the correct-answer diagonal. The quarter and tail views then separate dates and realized outcomes.

Groups defined by what happened next help explain mistakes afterward. They could not have been used to choose banks when the forecasts were issued. A lower average can coexist with serious misses in particular cases.



In [ ]:
READING_GUIDES['quarter_errors'] = 'Compare models within each predictor quarter. The three dates are shared historical conditions, so thousands of bank rows do not amount to thousands of independent economic periods.'
READING_GUIDES['tail_errors'] = 'Compare errors after grouping banks by realized growth. Those outcome groups are known afterward. This chart diagnoses mistakes and cannot show what we knew before the quarter ended.'
# Check errors by quarter and by realized outcome before interpreting an average.
question_card(
    title="Where does the average score hide the largest misses?",
    theme=theme,
    body="Compare all rows with each quarter’s realized bottom quartile and bottom decile, then inspect the quarter-by-quarter pattern.",
    kicker="Error diagnosis",
    chip_text="QUESTION",
)

# Keep time stability and weak-outcome severity as separate checks. One groups
# by quarter; the other groups by what actually happened inside each quarter.
quarter_rows = []
tail_rows = []

for quarter, part in result_frame.groupby("date"):
    for n in growth_predictions:
        quarter_rows.append(
            {"Quarter": quarter, "Model": n, "Rows": len(part), **scores(part.growth, part[n])}
        )

    # Include ties at the realized quantile boundary and show resulting sample sizes.
    for label, mask in [
        ("All", pd.Series(True, index=part.index)),
        ("Realized bottom 25%", part.growth.le(part.growth.quantile(0.25))),
        ("Realized bottom 10%", part.growth.le(part.growth.quantile(0.10))),
    ]:
        for n in growth_predictions:
            for idx in part.index[mask]:
                tail_rows.append(
                    {
                        "Index": idx,
                        "Slice": label,
                        "Model": n,
                        "Absolute error": abs(part.loc[idx, "growth"] - part.loc[idx, n]),
                        "Squared error": (part.loc[idx, "growth"] - part.loc[idx, n]) ** 2,
                    }
                )

quarter_scores = ordered_rows(
    pd.DataFrame(quarter_rows),
    ["Quarter", "Model"],
    category_orders={"Model": MODEL_ORDER},
)
table(
    quarter_scores,
    "Three quarters, separate error checks",
    {"MAE (pp)": "{:.3f}", "RMSE (pp)": "{:.3f}"},
)
fig = px.line(
    quarter_scores,
    x="Quarter",
    y="MAE (pp)",
    color="Model",
    symbol="Model",
    markers=True,
    color_discrete_map=MODEL_COLORS,
)
fig.update_xaxes(tickvals=sorted(result_frame.date.unique()), tickformat="%b %Y")

In [ ]:
chart(
    fig,
    "quarter_errors",
    "Quarter-level errors expose variation hidden by the average",
    "Predictor quarters; each outcome is one quarter later",
)
equal_quarter = (
    quarter_scores.groupby("Model", sort=False, observed=True)["MAE (pp)"]
    .mean()
    .reset_index(name="Equal-quarter MAE (pp)")
)
equal_quarter = ordered_rows(
    equal_quarter,
    ["Model"],
    category_orders={"Model": MODEL_ORDER},
)
table(
    equal_quarter, "Give each evaluation quarter equal weight", {"Equal-quarter MAE (pp)": "{:.3f}"}
)
tail_errors = (
    pd.DataFrame(tail_rows)
    .groupby(["Slice", "Model"], sort=False)
    .agg(Rows=("Index", "size"), MAE=("Absolute error", "mean"), MSE=("Squared error", "mean"))
    .reset_index()
)
tail_errors["MAE (pp)"] = 100 * tail_errors["MAE"]
tail_errors["RMSE (pp)"] = 100 * np.sqrt(tail_errors["MSE"])
tail_errors = ordered_rows(
    tail_errors,
    ["Slice", "Model"],
    category_orders={
        "Slice": SLICE_ORDER,
        "Model": MODEL_ORDER,
    },
)
table(
    tail_errors[["Slice", "Model", "Rows", "MAE (pp)", "RMSE (pp)"]],
    "How wrong are forecasts when growth is weak?",
    {"MAE (pp)": "{:.3f}", "RMSE (pp)": "{:.3f}"},
)
tail_errors["Slice"] = pd.Categorical(
    tail_errors["Slice"],
    categories=SLICE_ORDER,
    ordered=True,
)

# A slope chart makes the direction visible: follow each model from all rows
# toward increasingly weak realized outcomes.
fig = px.line(
    tail_errors.sort_values("Slice"),
    x="Slice",
    y="MAE (pp)",
    color="Model",
    markers=True,
    text="MAE (pp)",
    category_orders={"Slice": SLICE_ORDER},
    color_discrete_map=MODEL_COLORS,
    hover_data=["Rows"],
)
fig.update_traces(texttemplate="%{text:.1f}", textposition="top center")
fig.update_xaxes(title="Realized outcome group")
fig.update_yaxes(title="Mean absolute error (percentage points)")

In [ ]:
chart(
    fig,
    "tail_errors",
    "Forecast errors rise as realized growth gets weaker",
    "Follow each model from all rows to the bottom quartile and bottom decile",
)
quarter_scores.to_csv(OUT / "quarter_scores.csv", index=False)
tail_errors.to_csv(OUT / "tail_errors.csv", index=False)
weak = tail_errors.loc[
    (tail_errors.Slice == "Realized bottom 10%") & (tail_errors.Model == "MLP")
].iloc[0]
takeaway(
    "Check the weak-outcome error before using the forecast",
    f"The MLP misses by {weak['MAE (pp)']:.3f} pp on average across {int(weak.Rows):,} bottom-decile bank-quarters, compared with {mae['MLP']:.3f} pp overall. These groups were identified using realized outcomes.",
)

## 12 · Which selected banks had weak growth?

**In plain terms:** **Bank A: 0%. Bank B: 0%. Bank C: 0%. Which goes first?** Those forecasts provide no ordering. Zero growth remains useful for testing numerical forecast accuracy.

Now sort banks from lowest to highest predicted growth and select the first 10%. After the next quarter, count how many selected banks actually belong to that quarter's lowest-growth 10%. **Precision** is that count divided by the number selected. **Random selection** answers how many we would expect with no informative ordering. These are two different baseline jobs.

In March, Ridge selected 454 banks and found 112 of the realized lowest-growth group; the network found 109. The random expectation is 45.44. Lower growth describes a relative deposit outcome. A person must investigate its reason.



In [ ]:
# Fix the quarterly review capacity before looking at the rankings.
question_card(
    title="So which selected banks actually had the lowest growth later?",
    theme=theme,
    body="Rank banks from predicted growth first. Then compare that list with the realized bottom decile inside the same quarter.",
    kicker="Identification",
    chip_text="QUESTION",
)

# Fix the review capacity at 10% within each quarter. Certificate number breaks
# ties so the result can be reproduced exactly.
ranking = []
for quarter, part in result_frame.groupby("date"):
    selected_count = int(np.ceil(0.10 * len(part)))
    realized_bottom = set(part.sort_values(["growth", "CERT"]).head(selected_count).CERT)

    for model_name in growth_predictions:
        predicted_bottom = set(part.sort_values([model_name, "CERT"]).head(selected_count).CERT)
        hits = len(realized_bottom & predicted_bottom)
        has_ranking_signal = part[model_name].nunique() > 1
        spearman = (
            part.growth.corr(part[model_name], method="spearman") if has_ranking_signal else np.nan
        )

        ranking.append(
            {
                "Quarter": quarter,
                "Model": model_name,
                "Banks": len(part),
                "Selected": selected_count,
                "Hits": hits if has_ranking_signal else np.nan,
                "Precision": hits / selected_count if has_ranking_signal else np.nan,
                "Ranking status": "Forecast ordering"
                if has_ranking_signal
                else "All predictions tied; no ranking signal",
                "Recall": hits / len(realized_bottom) if has_ranking_signal else np.nan,
                "Spearman": spearman,
                "Random expectation": selected_count / len(part),
            }
        )

ranking = pd.DataFrame(ranking)

In [ ]:
READING_GUIDES['ranking'] = 'Each model has one marker per predictor quarter. Farther right means more selected banks later belonged to the lowest-growth group. Hover shows hits and selections. The reference marks roughly 10% random precision.'
# One model gets one row; each quarter gets one marker. Hover retains the
# exact count behind the percentage, and the CSV retains every audit column.
ranked_models = ranking.loc[ranking.Model.ne("Zero growth")].copy()
ranked_models["Quarter label"] = pd.to_datetime(ranked_models.Quarter).dt.strftime("%b %Y")
quarter_colors = {
    "Mar 2024": "#0B6F75",
    "Jun 2024": "#3F6294",
    "Sep 2024": "#A86223",
}
quarter_symbols = {"Mar 2024": "circle", "Jun 2024": "diamond", "Sep 2024": "square"}

fig = go.Figure()
model_rows = {"Persistence": 2, "Ridge": 1, "MLP": 0}
quarter_offsets = {"Mar 2024": 0.16, "Jun 2024": 0, "Sep 2024": -0.16}

for quarter_label in ["Mar 2024", "Jun 2024", "Sep 2024"]:
    part = ranked_models.loc[ranked_models["Quarter label"].eq(quarter_label)]
    fig.add_trace(
        go.Scatter(
            x=part["Precision"],
            y=part["Model"].map(model_rows) + quarter_offsets[quarter_label],
            mode="markers",
            name=quarter_label,
            marker={
                "size": 17,
                "symbol": quarter_symbols[quarter_label],
                "color": quarter_colors[quarter_label],
                "line": {"color": "white", "width": 1},
            },
            customdata=part[["Model", "Hits", "Selected"]].to_numpy(),
            hovertemplate=(
                "%{customdata[0]}<br>"
                + quarter_label
                + "<br>%{customdata[1]} of %{customdata[2]} selected banks"
                + "<br>Precision: %{x:.1%}<extra></extra>"
            ),
        )
    )

fig.update_xaxes(
    title="Selected banks with realized bottom-decile growth",
    tickformat=".0%",
    range=[0, 0.30],
)
fig.update_yaxes(
    title="",
    tickvals=[0, 1, 2],
    ticktext=["MLP", "Ridge", "Persistence"],
    range=[-0.45, 2.45],
)
fig.add_vline(
    x=0.10,
    line_dash="dash",
    line_color="#343B43",
)
chart(
    fig,
    "ranking",
    "Ridge leads in March and June; the network leads in September",
    "Dashed line: about 10% by chance. Ridge leads Mar and Jun; MLP leads Sep.",
    height=520,
    legend_y=-0.47,
)
ranking.to_csv(OUT / "ranking.csv", index=False)
precision = ranking.loc[ranking.Model.eq("MLP"), "Precision"].mean()
takeaway(
    "Both feature models beat chance in these three quarters",
    "Ridge leads in March and June; MLP leads in September. "
    "Hover over each point for hits out of banks selected. "
    "Three reused quarters cannot establish which model would lead later.",
)

## 13 · Would unusual reports give us a better review list?

**In plain terms:** **Imagine a bank reports an unusual cash ratio. Is that enough to predict next quarter's deposit change?** An anomaly detector can identify a report that differs from familiar reports. Its usefulness for finding future low-growth outcomes still needs evaluation.

A later experiment could compare an anomaly-based list with the forecast-based list at the same capacity and on the same available banks. This project has not measured that anomaly detector's performance. An unusual report provides a reason to inspect the source and context.



In [ ]:
# EXEMPLAR: analytical-question
question_card(
    title="So would anomaly detection change who we review?",
    theme=theme,
    body=(
        "Possibly. First decide whether the job is to anticipate a future "
        "decline or to investigate an unusual report already in hand."
    ),
    kicker="Next decision",
    chip_text="QUESTION",
)

In [ ]:
# EXEMPLAR: decision-ledger
# Each row names the evidence available when an analyst makes the decision.
review_questions = pd.DataFrame(
    [
        {
            "Question": "Who may have weak growth next quarter?",
            "Method": "Forecast ranking",
            "Evidence here": "Tested on three 2024 quarters",
            "Next check": "Repeat on a later untouched period",
        },
        {
            "Question": "Whose report looks unusual today?",
            "Method": "Anomaly detection",
            "Evidence here": "Not tested",
            "Next check": "Review top-ranked cases and false alerts",
        },
        {
            "Question": "Does this bank need follow-up?",
            "Method": "Human source review",
            "Evidence here": "Reporting changes can mimic events",
            "Next check": "Check filings, entity changes, and context",
        },
    ]
)
table(
    review_questions,
    "Three questions, three different kinds of evidence",
    wrap_columns={
        "Question": 230,
        "Method": 150,
        "Evidence here": 190,
        "Next check": 230,
    },
)

In [ ]:
# EXEMPLAR: counterintuitive-boundary
wm_counterintuitive_card(
    title="An unusual report starts an investigation",
    theme=theme,
    why_misread=("A sudden deposit change can look like a warning signal."),
    ordinary_process=(
        "Mergers, name changes, and reporting differences can also make a "
        "bank look unusual. The validation audit found a same-certificate "
        "name change beside an extreme balance jump."
    ),
    conclusion_boundary=(
        "Test an anomaly score on information available at the review date. "
        "At a fixed review capacity, measure useful cases and false alerts "
        "against analyst-reviewed records before recommending it."
    ),
    kicker="Interpretation check",
    chip_text="CHECK",
)
takeaway(
    "Anomaly detection deserves a separate trial",
    "The current ranking found weak-growth banks better than chance in three "
    "historical quarters. That does not tell us whether an anomaly score would "
    "send analysts to better cases. Evaluate it with the same review capacity "
    "and a clear definition of a useful review.",
)

## 14 · What did we learn?

**In plain terms:** **The original network reduced RMSE while zero growth retained lower MAE.** That tells us the network's extra complexity helped one error criterion in this historical sample. It did not establish a clear advantage across the jobs we tested.

**Did sorting help us choose cases to examine?** Ridge and the MLP placed more eventual low-growth outcomes in the assumed 10% list than random selection in each of three quarters. The follow-up comparisons now ask how much size alone explains, how uncertainty affects the difference, and whether simple seasonal forecasts change the error comparison.



In [ ]:
# Build the conclusion from the saved scores and the three-quarter limit.
rmse = metrics.set_index("Model")["RMSE (pp)"]
assert mae.idxmin() == "Zero growth"
assert rmse.idxmin() == "MLP"
conclusion = (
    f"The MLP had higher mean absolute error than zero growth "
    f"({mae['MLP']:.3f} versus {mae['Zero growth']:.3f} pp MAE). "
    f"It had the lowest RMSE ({rmse['MLP']:.3f} pp), and its 99th-percentile "
    "absolute error was below zero growth. Its small edge over Ridge does not "
    "establish dependable nonlinear value. Anomaly detection was not evaluated; "
    "the masterclass evaluates TimesFM separately after this core conclusion. "
    "Training used 214,425 examples across 38 predictor quarters. Evaluation used 13,532 examples across three reused 2024 predictor quarters. "
    "All four comparisons: "
    + "; ".join(
        f"{model}: MAE {mae[model]:.3f}, RMSE {rmse[model]:.3f} pp" for model in MODEL_ORDER
    )
    + "."
)
takeaway("Does this experiment establish nonlinear value?", conclusion)
takeaway(
    "For a bank analyst: use the result to guide research",
    "Inspect the source reports and institution changes behind large forecast misses. These quarterly balances do not establish withdrawals, bank runs, or the benefit of an intervention.",
)
takeaway(
    "For the next experiment: earn genuinely new evidence",
    "Resolve reporting-vintage and release-date availability, verify the historical form crosswalk, and evaluate a later period that has not guided development. Keep the strongest simple baseline.",
)
assert (
    source_hash
    == hashlib.sha256((ROOT / "data/fdic_financials_2013_2024.csv").read_bytes()).hexdigest()
)
assert raw_fingerprint == pd.util.hash_pandas_object(raw, index=True).sum()
assert np.allclose(rows.growth, (rows.next_deposits - rows.DEPDOM) / rows.DEPDOM)
assert np.allclose(rows.log_growth, np.log(rows.next_deposits / rows.DEPDOM))
import importlib.metadata

summary = {
    "source_sha256": source_hash,
    "target": "log growth; errors reported in ordinary percentage points",
    "history_gate": gate,
    "rows": {"train": len(train), "validation": len(valid), "holdout": len(test)},
    "winner_by_historical_MAE": winner,
    "mlp_minus_ridge_MAE_pp": difference,
    "conclusion": conclusion,
    "training_seconds": training_seconds,
    "frozen_plan": frozen,
    "packages": {
        p: importlib.metadata.version(p)
        for p in ["numpy", "pandas", "tensorflow", "scikit-learn", "plotly", "wm-notecards"]
    },
    "python": platform.python_version(),
}
(OUT / "run_summary.json").write_text(json.dumps(summary, indent=2))
print(
    "Audit passed: source unchanged, target arithmetic reconciled, predictions finite, results saved."
)

## Follow-up · Do simpler forecasts change the answer?

**In plain terms:** **Three extra baselines test whether the original comparison overlooked simple information.** Predict the middle training growth, predict the middle growth for the same calendar quarter, and fit Ridge using deposit size alone. Each uses earlier data for fitting. These follow-up analyses were developed after examining 2024.

The original result remains identifiable. The expanded scorecard answers whether those additional rules change the comparison on the same historical rows. Calendar patterns motivate a seasonal rule; they do not by themselves establish dependable forecasting skill.

**What does 21.9% precision mean?** About 22 out of every 100 selected banks later belonged to the lowest-growth group. Lift divides that share by the share expected from random selection. The exact random share is the rounded bottom-group count divided by the quarter's bank count, approximately 10%.

**How certain is a small difference?** We repeatedly resample banks, carrying their quarterly records together. We recalculate errors and precision on the existing forecasts and lists. The intervals describe variation across those banks, conditional on the fitted models and three 2024 dates. New periods and retraining bring additional uncertainty.




In [ ]:
# This source is copied into both notebooks as visible code cells.

In [ ]:
# Fit two fixed-rule forecasts using earlier outcomes only.
# A median is the middle training value. The seasonal rule uses one per quarter.
training_median = float(train["log_growth"].median())
quarter_medians = train.groupby(train["date"].dt.quarter)["log_growth"].median()
assert set(quarter_medians.index) == {1, 2, 3, 4}

# Bank size gets its own benchmark so we can assess the five-feature comparison.
size_scaler = StandardScaler().fit(train[["log_deposits"]])
size_train = size_scaler.transform(train[["log_deposits"]])
size_valid = size_scaler.transform(valid[["log_deposits"]])
size_model, size_alpha, size_tuning = choose_ridge(
    size_train,
    y_train,
    size_valid,
    y_valid,
    settings["ridge_alphas"],
)
size_tuning.to_csv(OUT / "size_only_validation.csv", index=False)


def followup_forecasts(frame):
    """Use the already-fitted rules; this function never learns from frame."""
    return {
        "Training median": np.full(len(frame), np.expm1(training_median)),
        "Seasonal median": np.expm1(frame["date"].dt.quarter.map(quarter_medians)),
        "Size-only Ridge": np.expm1(
            size_model.predict(size_scaler.transform(frame[["log_deposits"]]))
        ),
    }

In [ ]:
READING_GUIDES['followup_errors'] = 'Each dot is MAE on the same reused-2024 rows. Teal identifies the additional earlier-data-fitted rules. Seasonal median has the lowest observed MAE in this expanded comparison.'
# Give validation and historical evaluation their own rows. No setting is selected here.
followup_score_rows = []
for period, frame in [("Validation", valid), ("Reused 2024", holdout)]:
    predictions_for_period = followup_forecasts(frame)
    for name, prediction in predictions_for_period.items():
        followup_score_rows.append(
            {
                "Period": period,
                "Model": name,
                "Rows": len(frame),
                **scores(frame["growth"], prediction),
            }
        )
followup_scores = pd.DataFrame(followup_score_rows)
followup_scores.to_csv(OUT / "followup_scores.csv", index=False)
display(followup_scores.round(3))

extended = result_frame.copy()
for name, prediction in followup_forecasts(holdout).items():
    extended[name] = np.asarray(prediction)
extended.to_csv(OUT / "followup_predictions.csv", index=False)
followup_names = ["Training median", "Seasonal median", "Size-only Ridge"]
comparison_names = MODEL_ORDER + followup_names
combined_scores = pd.DataFrame(
    [{"Model": name, **scores(extended["growth"], extended[name])} for name in comparison_names]
).sort_values("MAE (pp)")
combined_scores.to_csv(OUT / "extended_scores.csv", index=False)

fig = go.Figure()
for _, row in combined_scores.iterrows():
    fig.add_trace(
        go.Scatter(
            x=[row["MAE (pp)"]],
            y=[row["Model"]],
            mode="markers+text",
            text=[f"{row['MAE (pp)']:.3f} pp"],
            textposition="middle right",
            showlegend=False,
            marker=dict(size=13, color="#0B6F75" if row["Model"] in followup_names else "#627381"),
        )
    )
fig.update_yaxes(autorange="reversed")
fig.update_xaxes(title="Mean absolute error (percentage points)", range=[3.3, 5.7])
chart(
    fig,
    "followup_errors",
    "The seasonal median improves historical MAE in a follow-up check",
    "Teal = follow-up comparison; all 13,532 reused-2024 rows; lower is better",
    height=600,
)
winning_followup = combined_scores.iloc[0]
takeaway(
    f"{winning_followup['Model']} has the lowest MAE in this expanded comparison",
    f"Its historical MAE is {winning_followup['MAE (pp)']:.3f} pp. "
    "The additional comparisons were developed after 2024 had been examined. "
    "The next evaluation should freeze these rules before new outcomes arrive.",
)

In [ ]:
# Pool training bank-quarters to answer: what fraction declined after each quarter?
# This differs from the earlier chart's median across separate years.
seasonal_rates = (
    train.assign(Quarter=train["date"].dt.quarter, Declined=train["growth"].lt(0))
    .groupby("Quarter")
    .agg(Rows=("CERT", "size"), Declines=("Declined", "sum"))
)
seasonal_rates["Decline share"] = seasonal_rates["Declines"] / seasonal_rates["Rows"]
seasonal_rates["Median log growth"] = quarter_medians
seasonal_rates["Median growth (%)"] = 100 * np.expm1(quarter_medians)
seasonal_rates.to_csv(OUT / "seasonal_rules.csv")
display(seasonal_rates.round(4))
takeaway(
    "Calendar timing deserves a frozen baseline in the next experiment",
    "The table pools bank-quarter observations within each predictor quarter of the year. "
    "The earlier dots keep individual years visible. A repeating pattern can motivate "
    "a forecast rule; performance on a later untouched period will test its usefulness.",
)

In [ ]:
READING_GUIDES['review_lift'] = 'A lift of 2 means twice the random concentration of the intended outcomes. Compare individual quarters and their equal-quarter means. The exact reference uses the rounded bottom-group prevalence.'
# Freeze the two sets before counting overlap. Equal sizes make precision = recall here.
def selection_evidence(frame, models):
    """Return per-quarter counts and row-level flags for the fitted ranking rules."""
    summaries = []
    flags = []
    for quarter, part in frame.groupby("date"):
        count = int(np.ceil(0.10 * len(part)))
        actual_ids = set(part.sort_values(["growth", "CERT"]).head(count)["CERT"])
        for name in models:
            # An identical forecast cannot meaningfully order banks within a quarter.
            if part[name].nunique() == 1:
                continue
            chosen_ids = set(part.sort_values([name, "CERT"]).head(count)["CERT"])
            selected = part["CERT"].isin(chosen_ids)
            hit = selected & part["CERT"].isin(actual_ids)
            prevalence = len(actual_ids) / len(part)
            precision_value = hit.sum() / count
            summaries.append(
                {
                    "Quarter": quarter,
                    "Model": name,
                    "Banks": len(part),
                    "Selected": count,
                    "Hits": int(hit.sum()),
                    "Precision": precision_value,
                    "Random expectation": prevalence,
                    "Expected random hits": count * prevalence,
                    "Lift": precision_value / prevalence,
                }
            )
            flags.append(
                pd.DataFrame(
                    {
                        "CERT": part["CERT"],
                        "Quarter": quarter,
                        "Model": name,
                        "Selected": selected.astype(int),
                        "Hit": hit.astype(int),
                    }
                )
            )
    return pd.DataFrame(summaries), pd.concat(flags, ignore_index=True)


ranking_models = ["Persistence", "Size-only Ridge", "Ridge", "MLP"]
followup_ranking, selection_flags = selection_evidence(extended, ranking_models)
followup_ranking.to_csv(OUT / "followup_ranking.csv", index=False)
selection_flags.to_csv(OUT / "selection_flags.csv", index=False)
mean_ranking = followup_ranking.groupby("Model")[["Precision", "Lift"]].mean()
mean_ranking = mean_ranking.sort_values("Lift", ascending=False)
mean_ranking.to_csv(OUT / "mean_ranking.csv")

fig = go.Figure(
    go.Scatter(
        x=mean_ranking["Lift"],
        y=mean_ranking.index,
        mode="markers+text",
        text=[f"{v:.2f}×" for v in mean_ranking["Lift"]],
        textposition="middle right",
        marker=dict(size=16, color="#0B6F75"),
    )
)
fig.add_vline(x=1, line_dash="dash", line_color="#707B87")
fig.update_xaxes(title="Concentration relative to random selection", range=[0.8, 2.7])
fig.update_yaxes(autorange="reversed")
chart(
    fig,
    "review_lift",
    "Five-feature rankings concentrate more cases than size alone",
    "Equal weight for each of three reused quarters; dashed line = random selection",
    height=520,
)
takeaway(
    "Bank size supplies useful ranking information in these quarters",
    f"Size-only Ridge: {mean_ranking.loc['Size-only Ridge', 'Precision']:.1%} precision. "
    f"Five-feature Ridge: {mean_ranking.loc['Ridge', 'Precision']:.1%}. "
    "This comparison measures association and a fitted selection rule. "
    "The paired intervals below show how much the difference varies across resampled banks.",
)

In [ ]:
READING_GUIDES['paired_uncertainty'] = 'A difference of zero means equal scores. The interval for MLP versus Ridge includes both orderings. Ridge minus size-only precision stays positive in this fixed-list bank-cluster resampling. Future quarters remain untested.'
# Resample banks, carrying each bank's repeated quarters together.
# Keep fitted models and the original review lists fixed: these are conditional intervals.
bank_ids = np.sort(extended["CERT"].unique())
quarters = sorted(extended["date"].unique())
selection_arrays = {}
for name in ranking_models:
    model_flags = selection_flags.loc[selection_flags["Model"].eq(name)]
    selection_arrays[name] = [
        model_flags.pivot(index="CERT", columns="Quarter", values=column)
        .reindex(index=bank_ids, columns=quarters)
        .fillna(0)
        .to_numpy()
        for column in ["Hit", "Selected"]
    ]

error_arrays = {}
for name in comparison_names:
    bank_errors = (
        extended.assign(Error=100 * (extended[name] - extended["growth"]).abs())
        .groupby("CERT")["Error"]
        .agg(["sum", "count"])
        .reindex(bank_ids)
    )
    error_arrays[name] = bank_errors.to_numpy()

rng = np.random.default_rng(42)
bootstrap_rows = []
for repeat in range(1000):
    multiplicity = rng.multinomial(len(bank_ids), np.full(len(bank_ids), 1 / len(bank_ids)))
    sampled_precision = {}
    for name, (hit_array, selected_array) in selection_arrays.items():
        quarter_hits = multiplicity @ hit_array
        quarter_selections = multiplicity @ selected_array
        sampled_precision[name] = np.mean(quarter_hits / quarter_selections)
    sampled_mae = {
        name: (multiplicity @ values[:, 0]) / (multiplicity @ values[:, 1])
        for name, values in error_arrays.items()
    }
    for first, second in [("Ridge", "Size-only Ridge"), ("MLP", "Ridge")]:
        bootstrap_rows.append(
            {
                "Repeat": repeat,
                "Metric": "Precision difference (pp)",
                "Comparison": f"{first} minus {second}",
                "Difference": 100 * (sampled_precision[first] - sampled_precision[second]),
            }
        )
    for first, second in [
        ("MLP", "Zero growth"),
        ("MLP", "Ridge"),
        ("Seasonal median", "Zero growth"),
    ]:
        bootstrap_rows.append(
            {
                "Repeat": repeat,
                "Metric": "MAE difference (pp)",
                "Comparison": f"{first} minus {second}",
                "Difference": sampled_mae[first] - sampled_mae[second],
            }
        )
bootstrap_samples = pd.DataFrame(bootstrap_rows)
bootstrap_samples.to_csv(OUT / "paired_bootstrap_samples.csv", index=False)

interval_rows = []

In [ ]:
score_lookup = combined_scores.set_index("Model")["MAE (pp)"]
for (metric_name, label), samples in bootstrap_samples.groupby(["Metric", "Comparison"]):
    first, second = label.split(" minus ")
    observed = (
        100 * (mean_ranking.loc[first, "Precision"] - mean_ranking.loc[second, "Precision"])
        if metric_name.startswith("Precision")
        else score_lookup[first] - score_lookup[second]
    )
    low, high = samples["Difference"].quantile([0.025, 0.975])
    interval_rows.append(
        {
            "Metric": metric_name,
            "Comparison": label,
            "Observed": observed,
            "95% lower": low,
            "95% upper": high,
            "Banks": len(bank_ids),
        }
    )
paired_intervals = pd.DataFrame(interval_rows)
paired_intervals.to_csv(OUT / "paired_intervals.csv", index=False)
display(paired_intervals.round(3))

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=["MAE: lower favors first model", "Precision: higher favors first model"],
    horizontal_spacing=0.32,
)
for column, metric_name in enumerate(["MAE difference (pp)", "Precision difference (pp)"], 1):
    for _, row in paired_intervals.loc[paired_intervals["Metric"].eq(metric_name)].iterrows():
        fig.add_trace(
            go.Scatter(
                x=[row["Observed"]],
                y=[row["Comparison"]],
                mode="markers",
                marker=dict(size=12, color="#0B6F75"),
                showlegend=False,
                error_x=dict(
                    type="data",
                    symmetric=False,
                    array=[row["95% upper"] - row["Observed"]],
                    arrayminus=[row["Observed"] - row["95% lower"]],
                ),
            ),
            row=1,
            col=column,
        )
    fig.add_vline(x=0, line_dash="dash", line_color="#707B87", row=1, col=column)
    fig.update_xaxes(title="Difference (percentage points)", row=1, col=column)
chart(
    fig,
    "paired_uncertainty",
    "Paired intervals show which historical differences remain uncertain",
    "1,000 paired bank resamples; fixed predictions and lists; three historical quarters",
    height=590,
)
takeaway(
    "Thousands of banks still share only three evaluation dates",
    "An interval crossing zero includes both directions of the measured difference. "
    "An interval excluding zero describes this conditional comparison. "
    "These resamples keep the 2024 environment and trained models fixed; "
    "a later untouched period is needed to test whether the result persists.",
)

In [ ]:
# Save a machine-readable account of what was fitted and when.
followup_design = {
    "status": "Follow-up analyses developed after examining 2024",
    "training_rows": len(train),
    "validation_rows": len(valid),
    "evaluation_rows": len(holdout),
    "training_last_outcome": str(train["target_date"].max()),
    "validation_last_outcome": str(valid["target_date"].max()),
    "training_median_log_growth": training_median,
    "seasonal_medians": quarter_medians.to_dict(),
    "size_ridge_alpha": size_alpha,
    "bootstrap": "1000 paired bank clusters; fixed fitted models and original selection lists",
    "average_ranking": "Arithmetic mean of three quarterly values; exact k/n random reference",
}
(OUT / "followup_design.json").write_text(json.dumps(followup_design, indent=2))

## Follow-up · What if we can examine 5%, 10%, or 20%?

**In plain terms:** **Imagine your team has half as much review time, or twice as much.** We change the number selected while keeping the desired outcome group fixed at the lowest-growth 10%. Precision measures the concentration in your list; recall measures how much of the entire target group your list reaches. This is a descriptive follow-up using the reused 2024 period.



In [ ]:
# Follow-up: change how many banks we review, holding the desired outcome group fixed.
# The bottom decile still contains ceil(10% * banks) at every review capacity.
capacity_rows = []
for quarter, quarter_rows in extended.groupby("date"):
    bank_count = len(quarter_rows)
    outcome_count = int(np.ceil(0.10 * bank_count))
    actual_lowest = set(quarter_rows.sort_values(["growth", "CERT"]).head(outcome_count).CERT)
    for capacity in [0.05, 0.10, 0.20]:
        selected_count = int(np.ceil(capacity * bank_count))
        for model in ["Persistence", "Size-only Ridge", "Ridge", "MLP"]:
            selected = set(quarter_rows.sort_values([model, "CERT"]).head(selected_count).CERT)
            hits = len(selected & actual_lowest)
            capacity_rows.append(
                {
                    "Quarter": quarter,
                    "Model": model,
                    "Capacity": capacity,
                    "Banks": bank_count,
                    "Outcome count": outcome_count,
                    "Selected": selected_count,
                    "Hits": hits,
                    "Precision": hits / selected_count,
                    "Recall": hits / outcome_count,
                    "Expected random hits": selected_count * outcome_count / bank_count,
                    "Lift": (hits / selected_count) / (outcome_count / bank_count),
                }
            )
capacity_results = pd.DataFrame(capacity_rows)
capacity_results.to_csv(OUT / "capacity_results.csv", index=False)
capacity_means = capacity_results.groupby(["Model", "Capacity"], as_index=False)[
    ["Precision", "Recall", "Lift"]
].mean()
capacity_means.to_csv(OUT / "capacity_means.csv", index=False)

In [ ]:
READING_GUIDES['capacity_check'] = 'Each panel changes review capacity from 5% to 20% while holding the lowest-growth outcome group at 10%. Lines are equal-quarter means. Precision describes the list, recall describes the target group, and lift compares concentration with random selection.'
# Precision asks about our list. Recall asks about all the outcomes we wanted to find.
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=[
        "Share of selections found",
        "Share of target group found",
        "Concentration vs random",
    ],
)
capacity_colors = {
    "Persistence": "#627381",
    "Size-only Ridge": "#76528B",
    "Ridge": "#3F6294",
    "MLP": "#0B6F75",
}
for model, frame in capacity_means.groupby("Model", sort=False):
    for column, metric in enumerate(["Precision", "Recall", "Lift"], start=1):
        fig.add_trace(
            go.Scatter(
                x=100 * frame.Capacity,
                y=frame[metric],
                mode="lines+markers",
                name=model,
                legendgroup=model,
                showlegend=column == 1,
                line_color=capacity_colors[model],
                hovertemplate=model
                + "<br>Review capacity: %{x}%<br>"
                + metric
                + ": %{y:.3f}<extra></extra>",
            ),
            row=1,
            col=column,
        )
        fig.update_xaxes(
            tickvals=[5, 10, 20], ticksuffix="%", title="Review capacity", row=1, col=column
        )
for column in [1, 2]:
    fig.update_yaxes(tickformat=".0%", rangemode="tozero", row=1, col=column)
fig.add_hline(y=1, line_dash="dash", line_color="#5E6871", row=1, col=3)
chart(
    fig,
    "capacity_check",
    "Larger review lists find more cases with lower concentration",
    "Equal-quarter means; fixed bottom-decile outcome; follow-up using three reused 2024 quarters",
    height=640,
)
display(capacity_means.round(3))
for capacity in [0.05, 0.10, 0.20]:
    row = capacity_means.loc[
        capacity_means.Model.eq("Ridge") & capacity_means.Capacity.eq(capacity)
    ].iloc[0]
    print(
        f"Ridge at {capacity:.0%} capacity: {row.Precision:.1%} of selections found the target; {row.Recall:.1%} of the target group was found; {row.Lift:.2f}x random concentration."
    )
takeaway(
    "More review slots change two different fractions",
    "Precision asks: of the banks we selected, how many finished in the lowest-growth group? "
    "Recall asks: of all banks in that group, how many did we select? "
    "Inspect both before deciding whether additional review time is worthwhile. "
    "We retain the original 10% assumption and do not select a new capacity from these reused outcomes.",
)

## Follow-up · What do the merger and liquidation change?

**In plain terms:** **The September-to-December 2023 Plus International jump spans a verified merger.** New York’s regulator records **October 1, 2023** as the effective date of Emigrant Bank merging into Plus International, under the Emigrant name. The bulletin was published in November 2024, so this source supports retrospective annotation. [Effective-date record](https://www.dfs.ny.gov/reports-and-publications/weekly-bulletins/wb20241122).

Silvergate announced its wind-down on **March 8, 2023** and reported fewer than **$10,000** of remaining deposit liabilities on **November 22, 2023**. These dates define our bounded wind-down sensitivity window. [March announcement](https://www.sec.gov/Archives/edgar/data/1312109/000131210923000058/ex991sipressrelease3x8x23.htm), [November repayment announcement](https://www.sec.gov/Archives/edgar/data/1312109/000095015723001160/ex99-1.htm).

Keep all eligible rows in the primary scores. Then omit intervals overlapping these two verified cases and recalculate using unchanged forecasts. This is a sensitivity to a small documented event list, with no claim that all mergers or closures have been identified. The reported balance change can combine depositor activity and institutional restructuring.




In [ ]:
READING_GUIDES['merger_timeline'] = 'The blue endpoints are reported domestic-deposit balances. The amber diamond marks the official effective merger date, one day after the first report. Actual calendar spacing keeps the event close to September 30.'
# EXEMPLAR: formula-card
# Read the reported balances and official effective date before interpreting the jump.
event_registry = pd.read_csv(ROOT / "sources/structural_events.csv")
merger_date = pd.Timestamp(
    event_registry.loc[event_registry.CERT.eq(57083), "Window start"].iloc[0]
)
merger_row = valid.loc[valid.CERT.eq(57083) & valid.date.eq(pd.Timestamp("2023-09-30"))].iloc[0]
assert merger_row.date < merger_date <= merger_row.target_date
merger_receipt = pd.DataFrame(
    [
        {
            "CERT": 57083,
            "Predictor date": merger_row.date,
            "Effective merger": merger_date,
            "Outcome date": merger_row.target_date,
            "Starting deposits USD": merger_row.DEPDOM * 1000,
            "Next deposits USD": merger_row.next_deposits * 1000,
            "Reported growth (%)": 100 * merger_row.growth,
        }
    ]
)
merger_receipt.to_csv(OUT / "merger_timeline.csv", index=False)
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=[merger_row.date, merger_row.target_date],
        y=[0, 0],
        mode="lines+markers",
        line=dict(color="#AAB9BC", width=4),
        marker=dict(size=13, color="#3F6294"),
        showlegend=False,
        hovertemplate="Report date: %{x|%b %d, %Y}<extra></extra>",
    )
)
fig.add_trace(
    go.Scatter(
        x=[merger_date],
        y=[0],
        mode="markers",
        marker=dict(size=16, color="#A86223", symbol="diamond"),
        showlegend=False,
        hovertemplate="Merger effective: %{x|%b %d, %Y}<extra></extra>",
    )
)
fig.add_annotation(
    x=merger_date,
    y=0,
    text="OCT 1, 2023<br>Merger effective",
    showarrow=True,
    ax=120,
    ay=-90,
    arrowcolor="#A86223",
    font=dict(color="#A86223", size=17),
)

In [ ]:
for date, label, anchor in [
    (merger_row.date, f"SEP 30<br>USD {merger_row.DEPDOM / 1000:.2f}M", "left"),
    (merger_row.target_date, f"DEC 31<br>USD {merger_row.next_deposits / 1000000:.2f}B", "right"),
]:
    fig.add_annotation(
        x=date,
        y=-0.25,
        text=label,
        showarrow=False,
        xanchor=anchor,
        font=dict(size=19, color="#3F6294"),
    )
fig.update_xaxes(
    range=[merger_row.date - pd.Timedelta(days=5), merger_row.target_date + pd.Timedelta(days=5)],
    visible=False,
)
fig.update_yaxes(range=[-0.65, 1.1], visible=False)
chart(
    fig,
    "merger_timeline",
    f"Reported deposit growth: +{100 * merger_row.growth:,.0f}%",
    "What happened between these reports? Plus International, CERT 57083; actual date spacing",
    height=570,
)
takeaway(
    "The forecast interval contains a merger",
    "Emigrant merged into Plus International under the Emigrant name on October 1, 2023, "
    "one day after the September report date. New York DFS published this record in November 2024. "
    "It supports a retrospective event annotation. Our five inputs contain no event flag; "
    "that does not establish what public information a person could have found at forecast time. "
    "The timeline supplies structural context without attributing every dollar of growth to the merger.",
)

In [ ]:
READING_GUIDES['structural_sensitivity'] = 'Compare unchanged forecasts on all eligible rows with the subset omitting the documented event intervals. The event list flags validation rows and no 2024 rows. Its incomplete coverage limits the interpretation.'
# Official event dates are annotations for a retrospective sensitivity.
# They are never fed into a forecast or used to choose model settings.
events = pd.read_csv(
    ROOT / "sources/structural_events.csv",
    parse_dates=["Window start", "Window end"],
)
display(events[["CERT", "Event", "Window start", "Window end", "Date meaning"]])


def known_event_mask(frame):
    """Flag a bank's forecast interval when it overlaps a verified event window."""
    flagged = pd.Series(False, index=frame.index)
    for _, event in events.iterrows():
        flagged |= (
            frame["CERT"].eq(event["CERT"])
            & frame["date"].lt(event["Window end"])
            & frame["target_date"].ge(event["Window start"])
        )
    return flagged


validation_frame = valid[
    ["CERT", "NAME", "date", "target_date", "DEPDOM", "next_deposits", "growth"]
].copy()
for name, prediction in validation_predictions.items():
    validation_frame[name] = np.expm1(prediction)

sensitivity_rows = []
flagged_examples = []
for period, frame in [("Validation", validation_frame), ("Reused 2024", result_frame)]:
    flagged = known_event_mask(frame)
    flagged_examples.append(frame.loc[flagged].assign(Period=period))
    for population, keep in [
        ("All eligible rows", pd.Series(True, index=frame.index)),
        ("Verified event intervals omitted", ~flagged),
    ]:
        for name in MODEL_ORDER:
            sensitivity_rows.append(
                {
                    "Period": period,
                    "Population": population,
                    "Model": name,
                    "Rows": int(keep.sum()),
                    "Flagged event rows": int(flagged.sum()),
                    **scores(frame.loc[keep, "growth"], frame.loc[keep, name]),
                }
            )
structural_scores = pd.DataFrame(sensitivity_rows)
structural_scores.to_csv(OUT / "structural_sensitivity.csv", index=False)
flagged_examples = pd.concat(flagged_examples, ignore_index=True)
flagged_examples.to_csv(OUT / "verified_event_rows.csv", index=False)
display(structural_scores.round(3))

fig = px.scatter(
    structural_scores.loc[structural_scores["Period"].eq("Validation")],
    x="RMSE (pp)",
    y="Model",
    color="Population",
    symbol="Population",
    hover_data=["Rows", "Flagged event rows"],
    log_x=True,
    color_discrete_sequence=["#A86223", "#0B6F75"],
)
fig.update_traces(marker_size=13)
chart(
    fig,
    "structural_sensitivity",
    "Verified structural events change the validation error comparison",
    "Fixed models; logarithmic RMSE axis; the primary scores retain every eligible row",
    height=530,
)
takeaway(
    "Reported deposit changes can include institutional restructuring",
    "The two-source event list identifies a merger interval and a bounded liquidation period. "
    "It is incomplete and was assembled retrospectively. Unchanged 2024 scores mean "
    "these particular windows flag no 2024 rows; other structural events may remain.",
)

In [ ]:
# Classify reporting gaps by observable dates. Their economic causes remain unknown.
comparable_panel = panel.loc[conditions["Insured domestic bank; comparable Call Report"]].copy()
comparable_panel["Prior report status"] = np.select(
    [
        comparable_panel["quarter_number"].eq(panel["quarter_number"].min()),
        comparable_panel["prior_quarter"].isna(),
        comparable_panel["quarter_number"].sub(comparable_panel["prior_quarter"]).ne(1),
    ],
    ["Dataset start", "First observed bank report", "Gap after earlier report"],
    default="Adjacent prior report",
)
gap_rows = []
for side, frame, column in [
    ("Prior", comparable_panel, "Prior report status"),
    (
        "Next after prior-adjacency filter",
        comparable_panel.loc[conditions["Adjacent prior quarter"]],
        "Next report status",
    ),
]:
    counts = frame.groupby(column).size().sort_values(ascending=False)
    for status, count in counts.items():
        gap_rows.append({"Stage": side, "Observed status": status, "Rows": count})
report_gaps = pd.DataFrame(gap_rows)
report_gaps.to_csv(OUT / "report_gap_classification.csv", index=False)
display(report_gaps)
takeaway(
    "An absent next report leaves the forecast without an observed answer",
    "Dataset endpoints, first or last appearances, and gaps are distinguishable from dates alone. "
    "Merger, closure, failure, and reporting explanations require additional records. "
    "This study evaluates bank-quarters with adjacent positive balances; institutions missing that outcome remain outside its scored population.",
)

## Next decision · What would a person investigate?

**In plain terms:** **A selected bank gives a human a place to begin asking questions.** Examine deposit composition, uninsured concentration, wholesale funding, available assets, borrowing capacity, pricing, and institution history. The current model uses five balance-sheet and deposit-history features. Its target is reported next-quarter deposit growth.

The [FFIEC’s UBPR](https://cdr.ffiec.gov/public/HelpFiles/FAQ.htm) compares a bank with its own history and peer banks. A future experiment could compare growth within documented historical peer groups. That changes the question and may change the size signal. We have not measured the benefit of that alternative. Peer definitions must match the historical period; current rules should not be silently applied to old observations.

**What would make the next evaluation stronger?** Freeze the candidate models, event rules, and selection capacity, then evaluate outcomes from a later period that has not guided development. Account for when reports actually became available; this dataset contains accounting dates and lacks historical publication vintages.

The agencies’ [April 2026 SR 26-2](https://www.federalreserve.gov/supervisionreg/srletters/SR2602.htm) replaced SR 11-7. Its benchmarking and outcome-analysis principles provide context; the Fed states it is most relevant to organizations above $30 billion. This class project makes no compliance claim.




## Deeper check · What changed in older reports?

**In plain terms:** **Two reports can use the same column name while meaning different things.** Imagine comparing a ratio collected under two reporting forms. First check how each form defines the fields. The older-report tables keep that unresolved comparability issue visible, alongside missing equity and report coverage.



In [ ]:
READING_GUIDES['missingness'] = 'These counts describe missing source entries in the broader audit population. Use the reporting-form receipt to see which groups contain the blanks before deciding how they affect the main population.'
READING_GUIDES['coverage_time'] = 'The height is the share of audit reports with missing equity at each date. The denominator is all reports in that audit quarter. Compare reporting forms before attributing a change to bank finances.'
# Reporting metadata lets us explain missing equity instead of treating it as random.
long_audit = long_history.merge(
    reporting,
    on=keys,
    how="left",
    validate="one_to_one",
    indicator=True,
)
coverage = (
    long_audit.groupby("REPDTE")
    .agg(
        Rows=("CERT", "size"),
        Equity_missing=("EQ", lambda values: values.isna().mean()),
        Metadata_matched=("_merge", lambda values: values.eq("both").mean()),
    )
    .reset_index()
)
coverage["Date"] = pd.to_datetime(coverage["REPDTE"].astype(str))
coverage.to_csv(OUT / "history_coverage.csv", index=False)

missing = long_history.isna().sum().rename("Missing rows").rename_axis("Field").reset_index()
missing = ordered_rows(
    missing,
    ["Missing rows", "Field"],
    ascending=[False, True],
)
table(missing, "Missing values in the 2010–2024 audit file")

fig = px.bar(
    missing,
    x="Missing rows",
    y="Field",
    orientation="h",
    color_discrete_sequence=["#A86223"],
)
chart(
    fig,
    "missingness",
    "Equity accounts for the missing financial values",
    "2010–2024 audit file; no values filled",
)

by_form = (
    long_audit.groupby(["BKCLASS", "CALLFORM"], dropna=False)
    .agg(
        Rows=("CERT", "size"),
        Missing_equity=("EQ", lambda values: values.isna().sum()),
    )
    .reset_index()
)
by_form.to_csv(OUT / "reporting_forms.csv", index=False)
table(
    ordered_rows(
        by_form.loc[by_form["Missing_equity"].gt(0)],
        ["Missing_equity", "BKCLASS", "CALLFORM"],
        ascending=[False, True, True],
    ),
    "Missing equity is concentrated in particular reporting groups",
)

In [ ]:
fig = px.line(
    coverage,
    x="Date",
    y="Equity_missing",
    markers=True,
    color_discrete_sequence=["#A86223"],
)
fig.update_yaxes(tickformat=".2%", title="Reports missing equity")
chart(
    fig,
    "coverage_time",
    "Missing equity varies across report dates",
    "All reports in the 2010–2024 audit file",
)

wm_counterintuitive_card(
    title="What a novice might overlook",
    theme=theme,
    why_misread=(
        "The column names match across fifteen years, and the newer extracts "
        "match the long file exactly."
    ),
    ordinary_process=(
        "A regulatory form can change while a downloaded field keeps the same "
        "name. The 2012 TFR-to-Call-Report conversion creates that risk."
    ),
    conclusion_boundary=(
        "Use the 48 quarters from 2013–2024. Keep 2010–2012 in the audit until "
        "a field-level historical crosswalk is verified."
    ),
    kicker="Comparability check",
    chip_text="LOOK TWICE",
)

## Next trial · Should the model change next quarter?

**In plain terms:** **First save a forecast before its answer is available.** That is the next evidence we need. A proposed shadow-mode trial would record lists while people continue their existing decision process.

1. **Freeze the rules:** model versions, earlier-data-fitted preprocessing, competing forecasts, review capacity, and event handling.
2. **Wait for reports to become available:** store publication/availability dates and the scoring timestamp, alongside accounting dates. This historical extract does not supply complete publication vintages.
3. **Count who can receive a forecast:** reports available, in-scope banks, usable features/history, scored banks, and reasons for exclusion. A future outcome must never determine who gets scored today.
4. **Save predictions and lists:** retain zero growth, training median, seasonal median, persistence, size-only Ridge, Ridge, and MLP as permanent forecast comparisons. Constants within a quarter provide no ranking signal.
5. **Wait for outcomes:** separately count which issued forecasts can now be evaluated and which remain unresolved. Measure forecast errors, review precision/recall/lift, and paired differences on matching populations.
6. **Investigate changes before retraining:** did coverage fall, inputs shift, errors grow, or a simpler method catch up? Did a merger explain a large miss? One unusual quarter and a persistent change call for different investigations.

**Should we retrain automatically?** New data alone supplies no evidence that replacing the model will help. Predefine review triggers, investigate what changed, and validate a proposed improvement before replacing the frozen system. The project has designed this next trial; it has not deployed or measured operational savings from it.

